# GALA glycosite classifier

Discriminates GALA-pathway from canonical Golgi O-GalNAc glycosites among observed glycoforms, using AlphaFold-derived structural features, UniProt topology and ESM-C residue embeddings.

**Reproducibility.** Place the Spectronaut reports and the two GlyGen CSV files in the same folder as this notebook and run cells top to bottom. AlphaFold structures, SASA, UniProt features and ESM-C embeddings are all built on the fly. Intermediate caches are written atomically so re-runs are incremental.

**Heavy compute.** ESM-C 600M and FreeSASA over ~1000 proteins take several hours on a single CUDA GPU.

**Required input files** (in the working directory):
- `2025_RB_Hek293_ERG1_5xLB_Report.xlsx`
- `June_2024_HeLa_GFP_ERG1_2LB_SSL_ChRed_Report_Pivot_notNorm_Summ.csv`
- `2025_RB_Huh7_ERG1_5xLB_Report.xlsx`
- `2025_RB_MHC_ERG1_7xLB_Report.xlsx`
- `human_proteoform_glycosylation_sites_glyconnect.csv` (download from https://data.glygen.org/datasets, dataset id `GLY_000150`)
- `human_proteoform_glycosylation_sites_unicarbkb.csv` (download from https://data.glygen.org/datasets, dataset id `GLY_000149`)

**Optional input files** (only needed for the validation sections at the end):
- `KPC47_WTG1_WT_ERG1_4LB_SSL_ChRed_Report_Pivot_notNorm_Summ.csv` (mouse)
- `2025_RB_KIC_HMP_10xLB_Report.xlsx` (mouse)

**Python dependencies**: `numpy`, `pandas`, `matplotlib`, `scipy`, `scikit-learn`, `requests`, `tqdm`, `biopython`, `freesasa>=2`, `catboost`, `joblib`, `shap`. For the M3 model with ESM-C, also `torch` (CUDA) and the `esm` package from EvolutionaryScale.

## 1. Setup

In [ ]:
import os
import re
import sys
import json
import math
import time
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import requests
from scipy import stats as scipy_stats
from scipy.spatial import cKDTree
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------
# Paths. By default the Spectronaut reports are expected in the
# working directory. Adjust RAW_DATA_DIR if they live elsewhere.
# ----------------------------------------------------------------
RAW_DATA_DIR = Path(".")
OUT_DIR      = Path("./gala_pipeline")

CACHE_DIR    = OUT_DIR / "cache"
TABLE_DIR    = OUT_DIR / "tables"
PLOT_DIR     = OUT_DIR / "plots"
MODEL_DIR    = OUT_DIR / "models"
AF_PDB_DIR   = OUT_DIR / "alphafold_pdb"
UNIPROT_DIR  = OUT_DIR / "uniprot_json"
ESMC_DIR     = OUT_DIR / "esmc_cache"
HF_DIR       = OUT_DIR / "hf_models"
GLYGEN_DIR   = OUT_DIR / "glygen"

for d in [CACHE_DIR, TABLE_DIR, PLOT_DIR, MODEL_DIR,
          AF_PDB_DIR, UNIPROT_DIR, ESMC_DIR, HF_DIR, GLYGEN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

# ----------------------------------------------------------------
# Atomic pickle write to avoid leaving zombie caches on interruption
# ----------------------------------------------------------------
def atomic_pickle_dump(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "wb") as f:
        pickle.dump(obj, f)
    tmp.replace(path)


def load_pickle_if_valid(path):
    """Load a pickle and discard it if it deserialises to an empty mapping."""
    path = Path(path)
    if not path.exists():
        return None
    try:
        with open(path, "rb") as f:
            obj = pickle.load(f)
        if hasattr(obj, "__len__") and len(obj) == 0:
            path.unlink()
            print(f"  Removed zombie empty cache: {path.name}")
            return None
        return obj
    except Exception:
        path.unlink()
        print(f"  Removed corrupted cache: {path.name}")
        return None


# ----------------------------------------------------------------
# Dependency and input check
# ----------------------------------------------------------------
REQUIRED_FILES = [
    "2025_RB_Hek293_ERG1_5xLB_Report.xlsx",
    "June_2024_HeLa_GFP_ERG1_2LB_SSL_ChRed_Report_Pivot_notNorm_Summ.csv",
    "2025_RB_Huh7_ERG1_5xLB_Report.xlsx",
    "2025_RB_MHC_ERG1_7xLB_Report.xlsx",
    "human_proteoform_glycosylation_sites_glyconnect.csv",
    "human_proteoform_glycosylation_sites_unicarbkb.csv",
]
OPTIONAL_FILES = [
    "KPC47_WTG1_WT_ERG1_4LB_SSL_ChRed_Report_Pivot_notNorm_Summ.csv",
    "2025_RB_KIC_HMP_10xLB_Report.xlsx",
]

REQUIRED_PACKAGES = [
    "numpy", "pandas", "matplotlib", "scipy", "sklearn",
    "requests", "tqdm", "Bio", "freesasa", "catboost", "joblib", "shap",
]
OPTIONAL_PACKAGES = ["torch", "esm"]

print(f"Working dir : {Path.cwd().resolve()}")
print(f"Raw data    : {RAW_DATA_DIR.resolve()}")
print(f"Pipeline out: {OUT_DIR.resolve()}")

print("\nInput files:")
missing_required = []
for fn in REQUIRED_FILES:
    p = RAW_DATA_DIR / fn
    flag = "OK" if p.exists() else "MISSING"
    print(f"  [{flag:7s}] {fn}")
    if not p.exists():
        missing_required.append(fn)
for fn in OPTIONAL_FILES:
    p = RAW_DATA_DIR / fn
    flag = "OK" if p.exists() else "absent (validation only)"
    print(f"  [{flag:7s}] {fn}")

print("\nPackages:")
missing_required_pkgs = []
for pkg in REQUIRED_PACKAGES:
    try:
        m = __import__(pkg)
        v = getattr(m, "__version__", "?")
        print(f"  [OK     ] {pkg:14s} {v}")
    except ImportError:
        print(f"  [MISSING] {pkg}")
        missing_required_pkgs.append(pkg)
for pkg in OPTIONAL_PACKAGES:
    try:
        m = __import__(pkg)
        v = getattr(m, "__version__", "?")
        print(f"  [OK     ] {pkg:14s} {v}")
    except ImportError:
        print(f"  [absent ] {pkg} (M3/ESM-C will be skipped)")

if missing_required:
    raise FileNotFoundError(
        f"Missing required input files in {RAW_DATA_DIR.resolve()}: {missing_required}"
    )
if missing_required_pkgs:
    raise ImportError(f"Missing required packages: {missing_required_pkgs}")

## 2. Load Spectronaut reports

Loads the four human training cell lines and drops rows where all intensity values are missing.

In [ ]:
CELL_LINE_CONFIG = {
    "HEK293": {
        "filename":      "2025_RB_Hek293_ERG1_5xLB_Report.xlsx",
        "erg1_pattern":  "Hek293_ERG1",
        "ctrl_pattern":  "Hek293_WT",
        "ctrl_exclude":  None,
        "drop_patterns": [],
    },
    "HeLa": {
        "filename":      "June_2024_HeLa_GFP_ERG1_2LB_SSL_ChRed_Report_Pivot_notNorm_Summ.csv",
        "erg1_pattern":  "HeLaERG1",
        "ctrl_pattern":  "HeLaGFP",
        "ctrl_exclude":  None,
        "drop_patterns": [],
    },
    "Huh7": {
        "filename":      "2025_RB_Huh7_ERG1_5xLB_Report.xlsx",
        "erg1_pattern":  "Huh7_ERG1",
        "ctrl_pattern":  "Huh7_WT",
        "ctrl_exclude":  None,
        "drop_patterns": ["lectin_TM"],
    },
    "MHC": {
        "filename":      "2025_RB_MHC_ERG1_7xLB_Report.xlsx",
        "erg1_pattern":  "MHC_ERG1",
        "ctrl_pattern":  "MHC_",
        "ctrl_exclude":  "ERG1",
        "drop_patterns": [],
    },
}

LOGFC_THRESHOLD = 3.0 # In the poster I used LOG2FC_threshold of 1 but you can "play" with this value. 
MIN_FRAC        = 0.5


def load_any(path):
    path = Path(path)
    if path.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(path)
    with open(path) as f:
        head = f.readline()
    sep = "," if head.count(",") > head.count(";") else ";"
    return pd.read_csv(path, sep=sep)


def identify_columns(df, erg1_pattern, ctrl_pattern, ctrl_exclude=None, drop_patterns=None):
    df = df.copy()
    if drop_patterns:
        for pat in drop_patterns:
            drop = [c for c in df.columns if pat in c]
            if drop:
                df = df.drop(columns=drop)
    erg1_cols = [c for c in df.columns if erg1_pattern in c and "MS2Quantity" in c]
    ctrl_cols = [c for c in df.columns if ctrl_pattern in c and "MS2Quantity" in c]
    if ctrl_exclude:
        ctrl_cols = [c for c in ctrl_cols if ctrl_exclude not in c]
    return df, erg1_cols, ctrl_cols


def drop_allnan_intensity_rows(df, intensity_cols):
    sub = df[intensity_cols].apply(pd.to_numeric, errors="coerce")
    is_empty = sub.isna().all(axis=1) | (sub.fillna(0).sum(axis=1) == 0)
    return df[~is_empty].copy(), int(is_empty.sum())


raw_datasets = {}
intensity_cols_by_cl = {}

for cl, cfg in CELL_LINE_CONFIG.items():
    p = RAW_DATA_DIR / cfg["filename"]
    if not p.exists():
        raise FileNotFoundError(f"{cl}: {p}")
    raw = load_any(p)
    d, e_cols, c_cols = identify_columns(
        raw,
        erg1_pattern  = cfg["erg1_pattern"],
        ctrl_pattern  = cfg["ctrl_pattern"],
        ctrl_exclude  = cfg["ctrl_exclude"],
        drop_patterns = cfg["drop_patterns"],
    )
    d_clean, n_dropped = drop_allnan_intensity_rows(d, e_cols + c_cols)
    raw_datasets[cl] = d_clean
    intensity_cols_by_cl[cl] = {"erg1": e_cols, "ctrl": c_cols}
    print(f"{cl:7s} rows={len(d_clean):>6,}  ERG1={len(e_cols)}  CTRL={len(c_cols)}  dropped={n_dropped}")

## 3. Peptide classification

Counts O-glycan sites by summing the N inside every `[HexNAc(N)...]` tag. The substring count is wrong on multi-HexNAc compositions.

Each peptide row is then labelled as `ERG1_only`, `shared_enriched`, `shared_stable` or `low_confidence`. The final binary label is `ERG1 = ERG1_only + shared_enriched` and `Shared = shared_stable`.

In [ ]:
_HEXNAC_TAG_RE = re.compile(r"HexNAc(?:\((\d+)\))?")


def count_o_glycan_sites(mod_seq):
    if pd.isna(mod_seq):
        return 0
    total = 0
    for tag in re.findall(r"\[([^\]]+)\]", str(mod_seq)):
        m = _HEXNAC_TAG_RE.search(tag)
        if m:
            total += int(m.group(1)) if m.group(1) else 1
    return total


def count_st(seq):
    if pd.isna(seq):
        return 0
    return sum(1 for aa in str(seq).upper() if aa in ("S", "T"))


def classify_peptides(df, erg1_cols, ctrl_cols,
                      min_frac=MIN_FRAC, logfc_threshold=LOGFC_THRESHOLD):
    df = df.copy()
    min_erg1 = math.ceil(len(erg1_cols) * min_frac)
    min_ctrl = math.ceil(len(ctrl_cols) * min_frac)
    df["present_ERG1"] = df[erg1_cols].notna().sum(axis=1) >= min_erg1
    df["present_CTRL"] = df[ctrl_cols].notna().sum(axis=1) >= min_ctrl
    df["mean_ERG1"]    = df[erg1_cols].mean(axis=1)
    df["mean_CTRL"]    = df[ctrl_cols].mean(axis=1)
    pseudo = 1.0
    df["log2FC"] = np.log2(
        (df["mean_ERG1"].fillna(0) + pseudo) /
        (df["mean_CTRL"].fillna(0) + pseudo)
    )
    cond = [
        df["present_ERG1"] & ~df["present_CTRL"],
        df["present_ERG1"] &  df["present_CTRL"] & (df["log2FC"] >  logfc_threshold),
        df["present_ERG1"] &  df["present_CTRL"] & (df["log2FC"] <= logfc_threshold),
    ]
    df["condition"] = np.select(cond,
        ["ERG1_only", "shared_enriched", "shared_stable"],
        default="low_confidence")
    return df


classified = {}
for cl, d in raw_datasets.items():
    e_cols = intensity_cols_by_cl[cl]["erg1"]
    c_cols = intensity_cols_by_cl[cl]["ctrl"]
    classified[cl] = classify_peptides(d, e_cols, c_cols)
    counts = classified[cl]["condition"].value_counts()
    print(f"{cl:7s} " + "  ".join(
        f"{cnd}={int(counts.get(cnd, 0))}"
        for cnd in ["ERG1_only", "shared_enriched", "shared_stable", "low_confidence"]
    ))

## 4. AlphaFold structures and SASA

Downloads AlphaFold v6 PDB files for every protein in the training set and computes per-residue SASA (FreeSASA) and pLDDT. Results are cached as `af_cache_human.pkl` so subsequent runs skip the download.

rSASA uses Tien et al. (2013) empirical normalisation on S and T residues only.

In [ ]:
import freesasa
from Bio.PDB import PDBParser

pdb_parser = PDBParser(QUIET=True)

TIEN_MAX = {
    "ALA": 129, "ARG": 274, "ASN": 195, "ASP": 193, "CYS": 167,
    "GLN": 225, "GLU": 223, "GLY": 104, "HIS": 224, "ILE": 197,
    "LEU": 201, "LYS": 236, "MET": 224, "PHE": 240, "PRO": 159,
    "SER": 155, "THR": 172, "TRP": 285, "TYR": 263, "VAL": 174,
}
AA3_TO_AA1 = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E",
    "GLY":"G","HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F",
    "PRO":"P","SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V",
}
AA1_TO_AA3 = {v: k for k, v in AA3_TO_AA1.items()}


def collect_training_accessions(classified):
    accs = set()
    for d in classified.values():
        for s in d["PG.ProteinAccessions"].dropna().astype(str):
            for tok in re.split(r"[;,]", s):
                tok = tok.strip()
                if tok:
                    accs.add(tok)
    return sorted(accs)


def download_alphafold(accessions, out_dir, version=6):
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    new, present, errors = 0, 0, []
    for acc in tqdm(accessions, desc="AlphaFold"):
        fp = out_dir / f"AF-{acc}-F1-model_v{version}.pdb"
        if fp.exists() and fp.stat().st_size > 0:
            present += 1
            continue
        try:
            url = f"https://alphafold.ebi.ac.uk/files/AF-{acc}-F1-model_v{version}.pdb"
            r = session.get(url, timeout=30)
            if r.status_code != 200:
                errors.append((acc, str(r.status_code)))
                continue
            fp.write_bytes(r.content)
            new += 1
            time.sleep(0.1)
        except Exception as e:
            errors.append((acc, repr(e)[:60]))
    return new, present, errors


def parse_pdb_sasa_plddt(pdb_path):
    """Return {aa3, sasa, plddt} dicts for chain A. Works with freesasa >= 2."""
    structure = pdb_parser.get_structure("af", str(pdb_path))
    model = next(structure.get_models())
    aa3_d, plddt_d = {}, {}
    for chain in model:
        for res in chain:
            if res.id[0] != " " or "CA" not in res:
                continue
            pos = int(res.id[1])
            aa3 = res.get_resname().upper()
            if aa3 not in AA3_TO_AA1:
                continue
            aa3_d[pos]   = aa3
            plddt_d[pos] = float(res["CA"].bfactor)
        break

    # freesasa functional API: freesasa.calc(structure)
    s = freesasa.Structure(str(pdb_path))
    result = freesasa.calc(s)
    chain_areas = result.residueAreas().get("A", {})

    sasa_d = {}
    for k, v in chain_areas.items():
        # Keys may be "12" or "12A" (insertion code). Keep only plain ints.
        try:
            pos = int(str(k).strip())
        except ValueError:
            continue
        sasa_d[pos] = float(v.total)

    return {"aa3": aa3_d, "sasa": sasa_d, "plddt": plddt_d}


AF_CACHE_PATH = CACHE_DIR / "af_cache_human.pkl"
af_cache = load_pickle_if_valid(AF_CACHE_PATH)
if af_cache is not None:
    print(f"Loaded af_cache: {len(af_cache):,} proteins")
else:
    af_cache = {}
    accessions = collect_training_accessions(classified)
    print(f"Unique proteins: {len(accessions):,}")
    new, present, errors = download_alphafold(accessions, AF_PDB_DIR)
    print(f"Downloaded: {new}, already present: {present}, errors: {len(errors)}")
    for acc in tqdm(accessions, desc="SASA+pLDDT"):
        fp = AF_PDB_DIR / f"AF-{acc}-F1-model_v6.pdb"
        if not fp.exists():
            continue
        try:
            rec = parse_pdb_sasa_plddt(fp)
            if len(rec["sasa"]) > 0:
                af_cache[acc] = rec
        except Exception:
            continue
    atomic_pickle_dump(af_cache, AF_CACHE_PATH)
    print(f"af_cache built: {len(af_cache):,} proteins")

if len(af_cache) == 0:
    raise RuntimeError(
        "af_cache is empty after build. Check network access to "
        "alphafold.ebi.ac.uk and that freesasa is installed correctly."
    )


def get_rsasa_plddt(uid, pos, aa1):
    rec = af_cache.get(uid, {})
    abs_sasa = rec.get("sasa",  {}).get(pos)
    plddt    = rec.get("plddt", {}).get(pos)
    if abs_sasa is None or pd.isna(abs_sasa):
        return None, plddt
    aa3 = AA1_TO_AA3.get(aa1)
    if aa3 not in ("SER", "THR"):
        return None, plddt
    return min(float(abs_sasa) / TIEN_MAX[aa3], 1.0), plddt

## 5. Top-N rSASA glycoform table

For each peptide row, ranks all S/T positions by rSASA and selects the top N most exposed, where N is the HexNAc count. Rows with `n_glycans > n_ST` (Spectronaut N-term mislocalisation artefacts) and multi-mapping peptides are dropped. Overlapping peptides on the same site set are deduplicated, keeping the longest.

In [ ]:
def first_token(s):
    if pd.isna(s):
        return None
    return re.split(r"[;,]", str(s))[0].strip()


EMPTY_GLY_COLS = [
    "cell_line", "uniprot_id", "gene", "peptide", "glycoform", "condition",
    "start_pos", "n_glycans", "n_ST", "log2FC",
    "sasa_topN", "sasa_all_mean", "plddt_topN", "topN_key", "is_unambiguous",
]


def build_glycoform_topn(d_cls, cl_label):
    df = d_cls.copy()
    df["n_glycans"] = df["EG.ModifiedSequence"].apply(count_o_glycan_sites)
    df["n_ST"]      = df["PEP.StrippedSequence"].apply(count_st)

    pos_str = df["PEP.PeptidePosition"].astype(str).str.strip()
    df["n_mappings"] = pos_str.str.split(r"[;,]").apply(
        lambda x: len([v for v in x if str(v).strip() not in ("", "nan", "None")])
    )
    df = df[df["n_mappings"] == 1].copy()
    df["start_pos"] = pd.to_numeric(df["PEP.PeptidePosition"], errors="coerce")
    df = df[df["start_pos"].notna()].copy()
    df["start_pos"] = df["start_pos"].astype(int)
    df = df[df["n_glycans"] <= df["n_ST"]].copy()
    df = df[df["n_glycans"] > 0].copy()

    records = []
    for _, row in df.iterrows():
        seq, start = row["PEP.StrippedSequence"], row["start_pos"]
        uid   = first_token(row["PG.ProteinAccessions"])
        n_gly = int(row["n_glycans"])
        if uid is None:
            continue
        st_info = []
        for i, aa in enumerate(str(seq)):
            if aa in ("S", "T"):
                pos = start + i
                rsasa, plddt = get_rsasa_plddt(uid, pos, aa)
                if rsasa is not None:
                    st_info.append({"pos": pos, "aa": aa, "rsasa": rsasa, "plddt": plddt})
        if not st_info:
            continue
        st_sorted = sorted(st_info, key=lambda x: x["rsasa"], reverse=True)
        n_pick = max(min(n_gly, len(st_sorted)), 1)
        top_n  = st_sorted[:n_pick]
        rsasa_vals = [x["rsasa"] for x in top_n]
        plddt_vals = [x["plddt"] for x in top_n if x["plddt"] is not None]
        records.append({
            "cell_line":     cl_label,
            "uniprot_id":    uid,
            "gene":          row.get("PG.Genes", ""),
            "peptide":       seq,
            "glycoform":     row["EG.ModifiedSequence"],
            "condition":     row["condition"],
            "start_pos":     start,
            "n_glycans":     n_gly,
            "n_ST":          int(row["n_ST"]),
            "log2FC":        row["log2FC"],
            "sasa_topN":     float(np.mean(rsasa_vals)),
            "sasa_all_mean": float(np.mean([x["rsasa"] for x in st_sorted])),
            "plddt_topN":    float(np.mean(plddt_vals)) if plddt_vals else np.nan,
            "topN_key":      tuple(sorted(x["pos"] for x in top_n)),
            "is_unambiguous": n_gly == int(row["n_ST"]),
        })
    if not records:
        return pd.DataFrame(columns=EMPTY_GLY_COLS)
    g = pd.DataFrame(records)
    g["topN_key_str"] = g["topN_key"].astype(str)
    g["pep_len"]      = g["peptide"].str.len()
    g = (g.sort_values("pep_len", ascending=False)
          .drop_duplicates(["uniprot_id", "n_glycans", "topN_key_str", "condition"],
                           keep="first")
          .drop(columns=["topN_key_str", "pep_len"])
          .reset_index(drop=True))
    return g


per_cl = {}
for cl, d_cls in classified.items():
    per_cl[cl] = build_glycoform_topn(d_cls, cl)
    print(f"{cl:7s} glycoforms after Top-N: {len(per_cl[cl]):>6,}")

all_glycoforms = pd.concat(per_cl.values(), axis=0, ignore_index=True)
if "condition" not in all_glycoforms.columns or len(all_glycoforms) == 0:
    raise RuntimeError(
        "No glycoforms could be built. Check that af_cache is populated "
        "and that PG.ProteinAccessions in the reports match the AF accessions."
    )
valid = all_glycoforms[
    all_glycoforms["condition"].isin(["ERG1_only", "shared_enriched", "shared_stable"])
].copy()
valid["group"] = valid["condition"].map({
    "ERG1_only":       "ERG1",
    "shared_enriched": "ERG1",
    "shared_stable":   "Shared",
})

print()
print(pd.crosstab(valid["cell_line"], valid["group"], margins=True).to_string())

## 6. GlyGen reference negatives

Loads the GlyConnect and UniCarbKB O-glycosylation site lists, filters to PubMed-curated entries on S/T residues, and deduplicates at site level.

**These two CSV files must be downloaded manually from the GlyGen portal**, because `data.glygen.org` is a JavaScript app and does not expose stable direct download URLs. Once downloaded, drop them anywhere the notebook can find them (working directory, `gala_pipeline/glygen/`, or `./glygen/`).

To download:
1. Go to https://data.glygen.org/datasets and search for `human_proteoform_glycosylation_sites_glyconnect`
2. Click the dataset, then download the CSV (current dataset id `GLY_000150`)
3. Repeat for `human_proteoform_glycosylation_sites_unicarbkb` (current dataset id `GLY_000149`)

The two file names expected by the notebook are:
- `human_proteoform_glycosylation_sites_glyconnect.csv`
- `human_proteoform_glycosylation_sites_unicarbkb.csv`

In [ ]:
GLYGEN_FILES = [
    "human_proteoform_glycosylation_sites_glyconnect.csv",
    "human_proteoform_glycosylation_sites_unicarbkb.csv",
]

# Search candidates in priority order
GLYGEN_SEARCH_DIRS = [
    GLYGEN_DIR,
    RAW_DATA_DIR,
    Path("./glygen"),
    Path("."),
]


def find_glygen_file(filename):
    for d in GLYGEN_SEARCH_DIRS:
        p = Path(d) / filename
        if p.exists() and p.stat().st_size > 50_000:
            return p
    return None


resolved = {}
missing = []
for fn in GLYGEN_FILES:
    p = find_glygen_file(fn)
    if p is None:
        missing.append(fn)
    else:
        resolved[fn] = p
        print(f"  Found {fn}: {p.resolve()} ({p.stat().st_size / 1024:.1f} KB)")

if missing:
    raise FileNotFoundError(
        "\n\nMissing GlyGen CSV files:\n  "
        + "\n  ".join(missing)
        + "\n\nDownload them manually from https://data.glygen.org/datasets\n"
        + "  - search 'human_proteoform_glycosylation_sites_glyconnect' (GLY_000150)\n"
        + "  - search 'human_proteoform_glycosylation_sites_unicarbkb'  (GLY_000149)\n"
        + "and place them in any of these folders:\n  "
        + "\n  ".join(str(Path(d).resolve()) for d in GLYGEN_SEARCH_DIRS)
    )

df_gc = pd.read_csv(resolved[GLYGEN_FILES[0]], dtype=str, keep_default_na=False)
df_uc = pd.read_csv(resolved[GLYGEN_FILES[1]], dtype=str, keep_default_na=False)
print(f"\nGlyConnect: {df_gc.shape}")
print(f"  columns: {list(df_gc.columns)[:10]} ...")
print(f"UniCarbKB : {df_uc.shape}")
print(f"  columns: {list(df_uc.columns)[:10]} ...")

# Quick sanity check: if pandas parsed the CSVs into a single huge column or
# rows are zero, the file is not a real GlyGen CSV (often an HTML download
# page).
for src_name, df in [("GlyConnect", df_gc), ("UniCarbKB", df_uc)]:
    if len(df) == 0 or len(df.columns) < 5:
        raise RuntimeError(
            f"{src_name} CSV looks invalid (rows={len(df)}, cols={len(df.columns)}). "
            "Re-download from data.glygen.org."
        )


def find_col(candidates, columns, exact=False):
    """Robust column lookup. Skips NaN/None column names."""
    cols_lower = {}
    for c in columns:
        if c is None:
            continue
        try:
            if pd.isna(c):
                continue
        except (TypeError, ValueError):
            pass
        cols_lower[str(c).lower()] = c
    for cand in candidates:
        cand_l = str(cand).lower()
        for col_l, col_orig in cols_lower.items():
            if (exact and cand_l == col_l) or (not exact and cand_l in col_l):
                return col_orig
    return None


UNIPROT_CANDIDATES = [
    "uniprotkb_canonical_ac", "uniprotkb_ac",
    "uniprot_canonical_ac", "uniprot_canonical", "uniprotkb",
    "uniprot_acc", "uniprot",
]
POSITION_CANDIDATES = [
    "glycosylation_site_uniprotkb", "glycosylation_site",
    "site_uniprotkb", "start_pos", "position", "site",
]
GLYCAN_TYPE_CANDIDATES = [
    "saccharide_subtype", "glycosylation_subtype",
    "glycan_subtype", "subtype",
    "glycosylation_type", "glycan_type",
    "saccharide",
]
AMINO_ACID_CANDIDATES = ["amino_acid", "residue", "site_aa", "aa"]


def normalise(df, source_name):
    col_uniprot     = find_col(UNIPROT_CANDIDATES,     df.columns)
    col_position    = find_col(POSITION_CANDIDATES,    df.columns)
    col_glycan_type = find_col(GLYCAN_TYPE_CANDIDATES, df.columns)
    col_amino_acid  = find_col(AMINO_ACID_CANDIDATES,  df.columns)
    col_xref_key    = find_col(["xref_key"], df.columns, exact=True)
    col_xref_id     = find_col(["xref_id"],  df.columns, exact=True)

    print(f"\n[{source_name}] detected columns:")
    for lbl, val in [("uniprot", col_uniprot), ("position", col_position),
                     ("glycan_type", col_glycan_type), ("amino_acid", col_amino_acid),
                     ("xref_key", col_xref_key), ("xref_id", col_xref_id)]:
        print(f"  {lbl:12s} -> {val!r}")

    if col_uniprot is None or col_position is None:
        raise RuntimeError(
            f"{source_name}: cannot locate UniProt or position column.\n"
            f"Available columns: {list(df.columns)}\n"
            f"Edit UNIPROT_CANDIDATES / POSITION_CANDIDATES above if needed."
        )

    return pd.DataFrame({
        "primary_accession": df[col_uniprot].astype(str).str.split("-").str[0].str.strip(),
        "residue_position":  pd.to_numeric(df[col_position], errors="coerce"),
        "glycan_type_raw":   df[col_glycan_type].astype(str) if col_glycan_type else "",
        "amino_acid":        df[col_amino_acid].astype(str)  if col_amino_acid  else "",
        "xref_key":          df[col_xref_key].astype(str)    if col_xref_key    else "",
        "xref_id":           df[col_xref_id].astype(str)     if col_xref_id     else "",
        "source":            source_name,
    })


df_raw = pd.concat([normalise(df_gc, "glyconnect"),
                    normalise(df_uc, "unicarbkb")], axis=0, ignore_index=True)
df_raw = df_raw.dropna(subset=["primary_accession", "residue_position"])
df_raw = df_raw[df_raw["primary_accession"].str.len() >= 6].copy()
df_raw["residue_position"] = df_raw["residue_position"].astype(int)

EXCLUDE = ["o-glcnac", "glcnac", "mannose", "fucose", "glucose", "n-link", "n-glyc"]
INCLUDE = ["o-link", "galnac", "mucin"]
gt = df_raw["glycan_type_raw"].str.lower()
df_g = df_raw[gt.apply(lambda s: any(p in s for p in INCLUDE)) &
              ~gt.apply(lambda s: any(p in s for p in EXCLUDE))].copy()

aa3_to_1 = {"SER": "S", "THR": "T", "TYR": "Y", "TRP": "W"}
df_g["residue_type"] = df_g["amino_acid"].str.upper().str.strip().replace(aa3_to_1)
df_g = df_g[df_g["residue_type"].isin(["S", "T", "", "?"])].copy()

PUBMED  = {"protein_xref_pubmed"}
CURATED = {"protein_xref_glyconnect", "protein_xref_unicarbkb_ds"}
xref = df_g["xref_key"].str.lower().str.strip()
df_g = df_g[xref.isin(PUBMED) | xref.isin(CURATED)].copy()

df_g["sort_key"] = (
    df_g["residue_type"].isin(["S", "T"]).astype(int) * 100
    + xref.loc[df_g.index].isin(PUBMED).astype(int) * 10
    + (df_g["source"] == "glyconnect").astype(int)
)
glygen_sites = (
    df_g.sort_values("sort_key", ascending=False)
        .drop_duplicates(["primary_accession", "residue_position"], keep="first")
        .drop(columns="sort_key")
        .reset_index(drop=True)
)[["primary_accession", "residue_position", "residue_type",
   "glycan_type_raw", "xref_key", "xref_id", "source"]]

print(f"\nGlyGen reference sites: {len(glygen_sites):,} on "
      f"{glygen_sites['primary_accession'].nunique():,} proteins")

## 7. rSASA distributions

Violin, CDF and exposure category panels comparing GALA, Golgi reference and GlyGen canonical sites.

In [ ]:
def lookup_rsasa_for_sites(sites_df):
    out, miss = [], 0
    for _, r in sites_df.iterrows():
        uid = r["primary_accession"]
        pos = int(r["residue_position"])
        aa1 = r["residue_type"] if r["residue_type"] in ("S", "T") else None
        if aa1 is None:
            aa3 = af_cache.get(uid, {}).get("aa3", {}).get(pos)
            aa1 = {"SER": "S", "THR": "T"}.get(aa3)
        if aa1 is None:
            miss += 1
            continue
        rsasa, plddt = get_rsasa_plddt(uid, pos, aa1)
        if rsasa is not None:
            out.append({"primary_accession": uid, "residue_position": pos,
                        "residue_type": aa1, "rsasa": rsasa, "plddt": plddt})
        else:
            miss += 1
    return pd.DataFrame(out), miss


glygen_with_sasa, miss = lookup_rsasa_for_sites(glygen_sites)
print(f"GlyGen with AF rSASA: {len(glygen_with_sasa):,} (missed {miss})")


C_GALA, C_GOLGI, C_GLYGEN = "#e74c3c", "#3498db", "#95a5a6"
C_BURIED, C_INTER, C_EXPOSE = "#2c3e50", "#e67e22", "#5dade2"


def cdf_xy(arr):
    arr = np.sort(arr)
    return arr, np.arange(1, len(arr) + 1) / len(arr)


def mwu(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.nan
    return scipy_stats.mannwhitneyu(a, b, alternative="two-sided").pvalue


def sasa_categories(arr):
    if len(arr) == 0:
        return 0.0, 0.0, 0.0
    n = len(arr)
    return ((arr < 0.2).sum()/n*100,
            ((arr >= 0.2) & (arr < 0.5)).sum()/n*100,
            (arr >= 0.5).sum()/n*100)


def fmt_p(p):
    if np.isnan(p):
        return "n/a"
    if p < 1e-300:
        return "p < 1e-300"
    return f"p = {p:.2e}"


gala   = valid.loc[valid["group"] == "ERG1",   "sasa_topN"].dropna().values
golgi  = valid.loc[valid["group"] == "Shared", "sasa_topN"].dropna().values
glygen = glygen_with_sasa["rsasa"].dropna().values

fig = plt.figure(figsize=(15, 4.8))
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.32)

# Violin
ax = fig.add_subplot(gs[0, 0])
data = [gala, golgi, glygen]
labels = [f"GALA\nn={len(gala):,}", f"Golgi\nn={len(golgi):,}", f"GlyGen\nn={len(glygen):,}"]
parts = ax.violinplot(data, showmeans=False, showmedians=False, showextrema=False, widths=0.85)
for pc, c in zip(parts["bodies"], [C_GALA, C_GOLGI, C_GLYGEN]):
    pc.set_facecolor(c); pc.set_edgecolor("black")
    pc.set_alpha(0.75); pc.set_linewidth(0.6)
for i, arr in enumerate(data, start=1):
    med, q1, q3 = np.median(arr), *np.percentile(arr, [25, 75])
    ax.hlines(med, i-0.28, i+0.28, colors="black", lw=2.0, zorder=5)
    ax.hlines([q1, q3], i-0.22, i+0.22, colors="black", lw=1.0, linestyles=(0, (3, 2)), zorder=5)
ax.axhline(0.2, color="gray", linestyle=(0, (2, 3)), lw=0.7, alpha=0.6)
ax.set_xticks([1, 2, 3]); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Top-N rSASA"); ax.set_ylim(-0.05, 1.2)
ax.set_title("Violin"); ax.grid(axis="y", alpha=0.2)
p_eg, p_gy, p_ey = mwu(gala, golgi), mwu(golgi, glygen), mwu(gala, glygen)
for x1, x2, ytop, txt in [(1, 2, 1.02, fmt_p(p_eg)),
                          (2, 3, 1.08, fmt_p(p_gy)),
                          (1, 3, 1.14, fmt_p(p_ey))]:
    ax.plot([x1, x1, x2, x2], [ytop, ytop+0.015, ytop+0.015, ytop], color="black", lw=0.8)
    ax.text((x1+x2)/2, ytop+0.025, txt, ha="center", va="bottom", fontsize=7.5)

# CDF
ax = fig.add_subplot(gs[0, 1])
for arr, lbl, c in [(gala,  f"GALA (n={len(gala):,})",  C_GALA),
                    (golgi, f"Golgi (n={len(golgi):,})", C_GOLGI),
                    (glygen,f"GlyGen (n={len(glygen):,})", C_GLYGEN)]:
    x, y = cdf_xy(arr)
    ax.plot(x, y, color=c, lw=1.8, label=lbl)
ax.axvline(0.2, color="gray", linestyle=(0, (2, 3)), lw=0.7, alpha=0.6)
ax.set_xlabel("Top-N rSASA"); ax.set_ylabel("Cumulative fraction")
ax.set_title("CDF"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.legend(loc="lower right", fontsize=8.5); ax.grid(alpha=0.2)

# Categories
ax = fig.add_subplot(gs[0, 2])
groups = ["GALA", "Golgi", "GlyGen"]
counts = [len(gala), len(golgi), len(glygen)]
bur, intm, exp = zip(*(sasa_categories(arr) for arr in (gala, golgi, glygen)))
xpos = np.arange(len(groups)); width = 0.6
bottom2 = np.array(bur) + np.array(intm)
ax.bar(xpos, bur,  width=width, color=C_BURIED, edgecolor="white", lw=0.6, label="Buried (<0.2)")
ax.bar(xpos, intm, width=width, bottom=bur,     color=C_INTER,  edgecolor="white", lw=0.6, label="Intermediate (0.2-0.5)")
ax.bar(xpos, exp,  width=width, bottom=bottom2, color=C_EXPOSE, edgecolor="white", lw=0.6, label="Exposed (>=0.5)")
for i, (b_, i_, e_) in enumerate(zip(bur, intm, exp)):
    if b_ > 4: ax.text(i, b_/2, f"{b_:.1f}%", ha="center", va="center", color="white", fontsize=8.5, fontweight="bold")
    if i_ > 4: ax.text(i, b_+i_/2, f"{i_:.1f}%", ha="center", va="center", color="white", fontsize=8.5, fontweight="bold")
    if e_ > 4: ax.text(i, b_+i_+e_/2, f"{e_:.1f}%", ha="center", va="center", color="white", fontsize=8.5, fontweight="bold")
ax.set_xticks(xpos)
ax.set_xticklabels([f"{n}\nn={c:,}" for n, c in zip(groups, counts)], fontsize=9)
ax.set_ylabel("% of sites"); ax.set_ylim(0, 105)
ax.set_title("Exposure categories"); ax.grid(axis="y", alpha=0.15)
ax.legend(loc="upper right", fontsize=8.5, framealpha=0.95)

fig.savefig(PLOT_DIR / "rsasa_distributions.png", dpi=150, bbox_inches="tight")
fig.savefig(PLOT_DIR / "rsasa_distributions.svg", bbox_inches="tight")
plt.show()

## 8. Site-level training set

Each glycoform is exploded into its component sites using `topN_key`. A site is positive (GALA) if any glycoform with that site falls into the ERG1 group. Sites observed both in GlyGen and as GALA candidates are relabelled as Golgi reference.

In [ ]:
def explode_topn(df):
    rows = []
    for _, r in df.iterrows():
        key = r["topN_key"]
        if isinstance(key, str):
            key = eval(key)
        for pos in key:
            rows.append({
                "uniprot_id":     r["uniprot_id"],
                "site":           int(pos),
                "group":          r["group"],
                "cell_line":      r["cell_line"],
                "is_unambiguous": r["is_unambiguous"],
            })
    return pd.DataFrame(rows)


site_explode = explode_topn(valid)


def aggregate(g):
    return pd.Series({
        "is_ERG1":              (g["group"] == "ERG1").any(),
        "is_Shared":            (g["group"] == "Shared").any(),
        "has_unambig_ERG1":     ((g["group"] == "ERG1") & g["is_unambiguous"]).any(),
        "n_glycoforms_ERG1":    int((g["group"] == "ERG1").sum()),
        "n_glycoforms_Shared":  int((g["group"] == "Shared").sum()),
        "cell_lines":           ",".join(sorted(g["cell_line"].unique())),
    })


site_consensus = (site_explode.groupby(["uniprot_id", "site"])
                   .apply(aggregate).reset_index()
                   .rename(columns={"uniprot_id": "primary_accession",
                                    "site": "residue_position"}))


def aa1_from_cache(uid, pos):
    aa3 = af_cache.get(uid, {}).get("aa3", {}).get(int(pos))
    return AA3_TO_AA1.get(aa3) if aa3 else None


site_consensus["residue_type"] = site_consensus.apply(
    lambda r: aa1_from_cache(r["primary_accession"], r["residue_position"]), axis=1)

erg1_pos = site_consensus.loc[site_consensus["is_ERG1"]].copy()
erg1_pos["target"] = 1
erg1_pos["source"] = "cellline_ERG1"

glygen_neg = glygen_sites[["primary_accession", "residue_position", "residue_type"]].copy()
glygen_neg["target"] = 0
glygen_neg["source"] = "GlyGen_canonical"
glygen_neg["has_unambig_ERG1"] = False
glygen_neg["n_glycoforms_ERG1"] = 0
glygen_neg["cell_lines"] = ""

mask_no_aa = ~glygen_neg["residue_type"].isin(["S", "T"])
if mask_no_aa.any():
    glygen_neg.loc[mask_no_aa, "residue_type"] = glygen_neg.loc[mask_no_aa].apply(
        lambda r: aa1_from_cache(r["primary_accession"], r["residue_position"]), axis=1)

# Overlap: ERG1+ sites that are also canonical GlyGen become negatives
erg1_pos["site_key"] = list(zip(erg1_pos["primary_accession"], erg1_pos["residue_position"].astype(int)))
glygen_neg["site_key"] = list(zip(glygen_neg["primary_accession"], glygen_neg["residue_position"].astype(int)))
overlap = set(erg1_pos["site_key"]) & set(glygen_neg["site_key"])
erg1_pos.loc[erg1_pos["site_key"].isin(overlap), "target"] = 0
erg1_pos.loc[erg1_pos["site_key"].isin(overlap), "source"] = "cellline_GlyGen_overlap"
glygen_neg = glygen_neg[~glygen_neg["site_key"].isin(overlap)].copy()

keep_cols = ["primary_accession", "residue_position", "residue_type",
             "target", "source", "has_unambig_ERG1", "n_glycoforms_ERG1", "cell_lines"]
training_sites = pd.concat([erg1_pos[keep_cols], glygen_neg[keep_cols]], ignore_index=True)
training_sites = training_sites[training_sites["residue_type"].isin(["S", "T"])]
training_sites = training_sites.drop_duplicates(
    ["primary_accession", "residue_position"]).reset_index(drop=True)

print(f"Training sites: {len(training_sites):,}")
print(f"  Positives (GALA):  {int((training_sites['target']==1).sum()):,}")
print(f"  Negatives (Golgi): {int((training_sites['target']==0).sum()):,}")
print(f"  Unique proteins:   {training_sites['primary_accession'].nunique():,}")
print()
print(training_sites["source"].value_counts().to_string())

## 9. Structural and topology features

Computes the 3D microenvironment (CA neighbour counts at 6/8/10 Angstrom, charge sums, charge imbalance) and the UniProt topology features (signal peptide, transmembrane, luminal/cytoplasmic).

`AF_SASA_Raw` and pLDDT are excluded as they are monotonic with rSASA on S/T residues. Protein-level features are excluded to avoid leakage by protein identity.

In [ ]:
charge_map = {"ASP": -1.0, "GLU": -1.0, "LYS": 1.0, "ARG": 1.0, "HIS": 0.1}

structural_3d_feature_cols = [
    "ca_neighbor_count_6A", "ca_neighbor_count_8A", "ca_neighbor_count_10A",
    "min_nonself_ca_dist", "mean_dist_5nearest_ca", "mean_neighbor_dist_8A",
    "n_charged_8A", "signed_charge_sum_8A", "abs_charge_sum_8A",
    "dist_weighted_charge_sum_8A", "invsq_charge_sum_8A", "invsq_abs_charge_sum_8A",
    "min_charged_dist_8A", "charge_imbalance_8A",
]

topology_feature_cols = [
    "Site_in_SignalPeptide", "Site_in_TM_Domain",
    "Site_is_Luminal_Extra", "Site_is_Cytoplasmic",
    "Dist_from_Signal_End",
    "Dist_to_Nearest_TM", "Rel_Pos_Mature_Protein",
    "Dist_to_TM_Boundary", "Rel_Pos_in_Luminal_Domain",
    "Dist_to_Luminal_Edge",
]


def parse_af_pdb_ca_table(pdb_path):
    structure = pdb_parser.get_structure("af", str(pdb_path))
    model = next(structure.get_models())
    records, seen = [], set()
    chains = sorted(list(model.get_chains()), key=lambda c: (c.id != "A", c.id))
    for chain in chains:
        for res in chain.get_residues():
            if res.id[0] != " " or "CA" not in res:
                continue
            resname = res.get_resname().upper()
            if resname not in AA3_TO_AA1:
                continue
            pos = int(res.id[1])
            if pos in seen:
                continue
            seen.add(pos)
            records.append({
                "residue_position": pos, "resname3": resname,
                "charge": charge_map.get(resname, 0.0),
                "x": float(res["CA"].coord[0]),
                "y": float(res["CA"].coord[1]),
                "z": float(res["CA"].coord[2]),
            })
    return pd.DataFrame(records)


def compute_3d_features(acc, sites_df, pdb_path):
    try:
        ca_df = parse_af_pdb_ca_table(pdb_path)
    except Exception:
        out = sites_df.copy()
        for c in structural_3d_feature_cols:
            out[c] = np.nan
        return out
    if ca_df.empty:
        out = sites_df.copy()
        for c in structural_3d_feature_cols:
            out[c] = np.nan
        return out
    coords    = ca_df[["x", "y", "z"]].values
    positions = ca_df["residue_position"].values.astype(int)
    charges   = ca_df["charge"].values.astype(float)
    pos_to_idx = {int(p): i for i, p in enumerate(positions)}
    tree = cKDTree(coords)
    rows = []
    for _, site in sites_df.iterrows():
        pos = int(site["residue_position"])
        row = {"primary_accession": acc, "residue_position": pos}
        if pos not in pos_to_idx:
            for c in structural_3d_feature_cols:
                row[c] = np.nan
            rows.append(row); continue
        idx = pos_to_idx[pos]; coord = coords[idx]
        k = min(len(coords), 6)
        dists, idxs = tree.query(coord, k=k)
        dists = np.atleast_1d(dists); idxs = np.atleast_1d(idxs)
        nonself = dists[idxs != idx]
        row["min_nonself_ca_dist"]   = float(np.min(nonself)) if len(nonself) else np.nan
        row["mean_dist_5nearest_ca"] = float(np.mean(nonself[:5])) if len(nonself) else np.nan
        for radius in [6.0, 8.0, 10.0]:
            nbrs = [j for j in tree.query_ball_point(coord, r=radius) if j != idx]
            row[f"ca_neighbor_count_{int(radius)}A"] = len(nbrs)
        nbrs8 = [j for j in tree.query_ball_point(coord, r=8.0) if j != idx]
        if nbrs8:
            d8 = np.linalg.norm(coords[nbrs8] - coord, axis=1)
            row["mean_neighbor_dist_8A"] = float(np.mean(d8))
            ch8 = charges[nbrs8]
            mask = ch8 != 0
            cc, cd = ch8[mask], d8[mask]
            row["n_charged_8A"] = int(len(cc))
            if len(cc) > 0:
                sd = np.maximum(cd, 1e-6)
                signed = float(np.sum(cc)); abssum = float(np.sum(np.abs(cc)))
                row["signed_charge_sum_8A"]        = signed
                row["abs_charge_sum_8A"]           = abssum
                row["dist_weighted_charge_sum_8A"] = float(np.sum(cc / sd))
                row["invsq_charge_sum_8A"]         = float(np.sum(cc / sd**2))
                row["invsq_abs_charge_sum_8A"]     = float(np.sum(np.abs(cc) / sd**2))
                row["min_charged_dist_8A"]         = float(np.min(cd))
                row["charge_imbalance_8A"]         = signed / abssum if abssum > 0 else 0.0
            else:
                for c in ["signed_charge_sum_8A", "abs_charge_sum_8A",
                          "dist_weighted_charge_sum_8A", "invsq_charge_sum_8A",
                          "invsq_abs_charge_sum_8A", "charge_imbalance_8A"]:
                    row[c] = 0.0
                row["min_charged_dist_8A"] = np.nan
        else:
            row["mean_neighbor_dist_8A"] = np.nan
            row["n_charged_8A"] = 0
            for c in ["signed_charge_sum_8A", "abs_charge_sum_8A",
                      "dist_weighted_charge_sum_8A", "invsq_charge_sum_8A",
                      "invsq_abs_charge_sum_8A", "charge_imbalance_8A"]:
                row[c] = 0.0
            row["min_charged_dist_8A"] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def fetch_uniprot_json(acc, sleep_sec=0.05, timeout=30):
    cache_path = UNIPROT_DIR / f"{acc}.json"
    if cache_path.exists():
        try:
            with open(cache_path) as f:
                return json.load(f)
        except Exception:
            pass
    url = f"https://rest.uniprot.org/uniprotkb/{acc}.json"
    try:
        r = requests.get(url, timeout=timeout)
        if r.status_code != 200:
            return None
        data = r.json()
        with open(cache_path, "w") as f:
            json.dump(data, f)
        time.sleep(sleep_sec)
        return data
    except Exception:
        return None


def parse_uniprot_topology(data):
    if data is None:
        return {"protein_length": np.nan, "signal_ranges": [], "tm_ranges": [],
                "luminal_extra_ranges": [], "cytoplasmic_ranges": []}
    plen = data.get("sequence", {}).get("length", np.nan)
    sig, tm, lum, cyt = [], [], [], []
    for feat in data.get("features", []):
        ft = str(feat.get("type", "")).strip().lower()
        desc = str(feat.get("description", "")).strip().lower()
        loc = feat.get("location", {})
        try:
            s = int(loc.get("start", {}).get("value"))
            e = int(loc.get("end", {}).get("value"))
        except (TypeError, ValueError):
            continue
        if ft in {"signal", "signal peptide"}:
            sig.append((s, e))
        elif ft == "transmembrane":
            tm.append((s, e))
        elif ft == "topological domain":
            if any(k in desc for k in ["extracellular", "lumenal", "luminal", "periplasmic"]):
                lum.append((s, e))
            elif any(k in desc for k in ["cytoplasmic", "cytosolic"]):
                cyt.append((s, e))
    return {"protein_length": plen, "signal_ranges": sig, "tm_ranges": tm,
            "luminal_extra_ranges": lum, "cytoplasmic_ranges": cyt}


def in_any_range(pos, ranges):
    return any(s <= pos <= e for s, e in ranges)
def min_distance_to_ranges(pos, ranges):
    if not ranges: return np.nan
    return float(min(0 if s <= pos <= e else (s - pos if pos < s else pos - e) for s, e in ranges))
def min_distance_to_range_boundary(pos, ranges):
    if not ranges: return np.nan
    bs = [b for s, e in ranges for b in (s, e)]
    return float(min(abs(pos - b) for b in bs))
def rel_pos_in_ranges(pos, ranges):
    for s, e in ranges:
        if s <= pos <= e:
            L = e - s + 1
            return 0.0 if L <= 1 else float((pos - s) / (L - 1))
    return np.nan
def dist_to_luminal_edge(pos, ranges):
    if not ranges: return np.nan
    for s, e in ranges:
        if s <= pos <= e:
            return float(min(pos - s, e - pos))
    return min_distance_to_ranges(pos, ranges)


# ----------------------------------------------------------------
# Run feature pipeline on training_sites
# ----------------------------------------------------------------
unique_proteins = sorted(training_sites["primary_accession"].unique())
print(f"Proteins to feature-compute: {len(unique_proteins):,}")

# AF rSASA at site level
af_rows = []
for _, r in training_sites.iterrows():
    uid, pos, aa = r["primary_accession"], int(r["residue_position"]), r["residue_type"]
    abs_sasa = af_cache.get(uid, {}).get("sasa", {}).get(pos)
    if abs_sasa is None or pd.isna(abs_sasa) or aa not in ("S", "T"):
        rsasa = np.nan
    else:
        rsasa = min(float(abs_sasa) / TIEN_MAX[AA1_TO_AA3[aa]], 1.0)
    af_rows.append({"primary_accession": uid, "residue_position": pos, "AF_RSA": rsasa})
af_feats = pd.DataFrame(af_rows)

# PDB index
pdb_index = {}
for pdb_path in AF_PDB_DIR.rglob("*.pdb"):
    m = re.search(r"AF-([A-Z0-9]+)-F1-model_v(\d+)\.pdb", pdb_path.name)
    if m:
        acc = m.group(1); ver = int(m.group(2))
        if acc not in pdb_index or ver > pdb_index[acc][1]:
            pdb_index[acc] = (pdb_path, ver)
pdb_index = {acc: p for acc, (p, _) in pdb_index.items()}

# 3D features
feat_3d_tables = []
for acc, g in tqdm(training_sites.groupby("primary_accession"),
                   total=training_sites["primary_accession"].nunique(),
                   desc="3D features"):
    if acc not in pdb_index:
        tmp = g[["primary_accession", "residue_position"]].copy()
        for c in structural_3d_feature_cols:
            tmp[c] = np.nan
        feat_3d_tables.append(tmp); continue
    feat_3d_tables.append(compute_3d_features(
        acc, g[["primary_accession", "residue_position"]], pdb_index[acc]))
feats_3d = pd.concat(feat_3d_tables, ignore_index=True)

# UniProt topology
parsed = {}
for acc in tqdm(unique_proteins, desc="UniProt"):
    parsed[acc] = parse_uniprot_topology(fetch_uniprot_json(acc))

topo_rows = []
for _, r in training_sites.iterrows():
    acc, pos = r["primary_accession"], int(r["residue_position"])
    p = parsed.get(acc, {"protein_length": np.nan, "signal_ranges": [], "tm_ranges": [],
                         "luminal_extra_ranges": [], "cytoplasmic_ranges": []})
    sig, tm, lum = p["signal_ranges"], p["tm_ranges"], p["luminal_extra_ranges"]
    cyt, plen = p["cytoplasmic_ranges"], p["protein_length"]
    signal_end = max([e for _, e in sig], default=np.nan)
    if pd.notna(signal_end):
        dist_signal = float(pos - signal_end); mature_start = int(signal_end) + 1
    else:
        dist_signal = np.nan; mature_start = 1
    rel_mature = (float((pos - mature_start) / max((plen - mature_start), 1))
                  if pd.notna(plen) and plen > 0 else np.nan)
    topo_rows.append({
        "primary_accession": acc, "residue_position": pos,
        "Site_in_SignalPeptide":     int(in_any_range(pos, sig)),
        "Site_in_TM_Domain":         int(in_any_range(pos, tm)),
        "Site_is_Luminal_Extra":     int(in_any_range(pos, lum)),
        "Site_is_Cytoplasmic":       int(in_any_range(pos, cyt)),
        "Dist_from_Signal_End":      dist_signal,
        "Dist_to_Nearest_TM":        min_distance_to_ranges(pos, tm),
        "Rel_Pos_Mature_Protein":    rel_mature,
        "Dist_to_TM_Boundary":       min_distance_to_range_boundary(pos, tm),
        "Rel_Pos_in_Luminal_Domain": rel_pos_in_ranges(pos, lum),
        "Dist_to_Luminal_Edge":      dist_to_luminal_edge(pos, lum),
    })
feats_topo = pd.DataFrame(topo_rows)

df_train = (training_sites
            .merge(af_feats, on=["primary_accession", "residue_position"], how="left")
            .merge(feats_3d, on=["primary_accession", "residue_position"], how="left")
            .merge(feats_topo, on=["primary_accession", "residue_position"], how="left"))
df_train = df_train[df_train["AF_RSA"].notna()].reset_index(drop=True)
print(f"df_train: {df_train.shape}  pos={int((df_train['target']==1).sum())}  neg={int((df_train['target']==0).sum())}")

## 10. ESM-C residue embeddings

Computes residue-level embeddings from ESM-C 600M for every site in the training set. Sequences are pulled from the cached UniProt JSONs. Proteins longer than 2500 residues are skipped to fit in VRAM.

In [ ]:
# Windows symlink workaround for ESM-C / HuggingFace
import huggingface_hub
import esm.utils.constants.esm3 as esm3_constants

hf_local_dir = OUT_DIR / "hf_models"
hf_local_dir.mkdir(parents=True, exist_ok=True)

_ESM_REPOS = {
    "EvolutionaryScale/esmc-300m-2024-12",
    "EvolutionaryScale/esmc-600m-2024-12",
    "EvolutionaryScale/esm3-sm-open-v1",
}
_orig_snap = huggingface_hub.snapshot_download

def _patched_snap(repo_id, *args, **kwargs):
    if repo_id in _ESM_REPOS:
        local = hf_local_dir / repo_id.replace("/", "__")
        local.mkdir(parents=True, exist_ok=True)
        kwargs.pop("local_dir", None)
        kwargs.pop("local_dir_use_symlinks", None)
        kwargs["local_dir"] = str(local)
        kwargs["local_dir_use_symlinks"] = False
        kwargs["max_workers"] = 1
    return _orig_snap(repo_id, *args, **kwargs)

# Patch the reference imported by esm at module-load time
esm3_constants.snapshot_download = _patched_snap
print(f"HF cache redirected to {hf_local_dir.resolve()}, symlinks disabled")

In [ ]:
import gc

ESMC_CACHE = ESMC_DIR / "esmc600_residue_embeddings_cache.pkl"
MAX_SEQ_LEN = 2500

embedding_cache = load_pickle_if_valid(ESMC_CACHE)
if embedding_cache is None:
    embedding_cache = {}
print(f"ESM-C cache: {len(embedding_cache):,} entries")

target_sites = df_train[["primary_accession", "residue_position"]].drop_duplicates().reset_index(drop=True)
target_keys = [f"{r.primary_accession}|{int(r.residue_position)}" for r in target_sites.itertuples()]
needed = [k for k in target_keys if k not in embedding_cache]
print(f"Need to compute: {len(needed):,} / {len(target_keys):,}")

if needed:
    import torch
    if not torch.cuda.is_available():
        print("CUDA not available. Skipping ESM-C compute. M3 will only use cached embeddings.")
    else:
        os.environ["HF_HOME"] = str(HF_DIR)
        from esm.models.esmc import ESMC
        from esm.sdk.api import ESMProtein, LogitsConfig

        needed_by_acc = {}
        for k in needed:
            acc, pos = k.rsplit("|", 1)
            needed_by_acc.setdefault(acc, []).append(int(pos))

        sequence_map = {}
        for acc in needed_by_acc:
            jp = UNIPROT_DIR / f"{acc}.json"
            if jp.exists():
                with open(jp) as f:
                    seq = json.load(f).get("sequence", {}).get("value")
                    if seq:
                        sequence_map[acc] = seq

        gc.collect(); torch.cuda.empty_cache()
        model = ESMC.from_pretrained("esmc_600m").to("cuda").half().eval()

        new_embs = 0
        for acc in tqdm(sorted(needed_by_acc, key=lambda a: len(sequence_map.get(a, ""))),
                        desc="ESM-C"):
            if acc not in sequence_map:
                continue
            seq = sequence_map[acc]; L = len(seq)
            if L > MAX_SEQ_LEN:
                continue
            try:
                with torch.no_grad():
                    pt  = model.encode(ESMProtein(sequence=seq))
                    out = model.logits(pt, LogitsConfig(sequence=True, return_embeddings=True))
                emb = out.embeddings
                if isinstance(emb, torch.Tensor):
                    emb = emb.detach().cpu()
                if emb.ndim == 3:
                    emb = emb.squeeze(0)
                emb = emb.float().numpy()
                nt = emb.shape[0]
                for pos in needed_by_acc[acc]:
                    if 1 <= pos <= L:
                        if nt >= L + 2:
                            idx = pos
                        elif nt == L:
                            idx = pos - 1
                        else:
                            continue
                        embedding_cache[f"{acc}|{pos}"] = emb[idx].astype(np.float32)
                        new_embs += 1
                torch.cuda.empty_cache()
            except Exception:
                torch.cuda.empty_cache(); gc.collect()
            if new_embs > 0 and new_embs % 500 == 0:
                atomic_pickle_dump(embedding_cache, ESMC_CACHE)
        atomic_pickle_dump(embedding_cache, ESMC_CACHE)
        del model; gc.collect(); torch.cuda.empty_cache()
        print(f"New embeddings: {new_embs:,}")

# Build per-site embedding df
rows, ESMC_DIM = [], None
for _, r in target_sites.iterrows():
    k = f"{r.primary_accession}|{int(r.residue_position)}"
    if k in embedding_cache:
        v = embedding_cache[k]
        if ESMC_DIM is None:
            ESMC_DIM = len(v)
        rec = {"primary_accession": r.primary_accession,
               "residue_position":  int(r.residue_position)}
        for i, x in enumerate(v):
            rec[f"ESMC600_{i}"] = float(x)
        rows.append(rec)
esmc_df = pd.DataFrame(rows) if rows else pd.DataFrame()
print(f"ESM-C site embeddings: {esmc_df.shape}")

## 11. Training matrices

- M1: rSASA only (1 feature)
- M2: rSASA + 14 3D/electrostatic + 10 topology (25 features)
- M3: M2 + ESM-C PCA to 64 components (89 features)

PCA is fit only on the M3 training rows to avoid distributional leakage to external sets.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import joblib

FEAT_X1 = ["AF_RSA"]
FEAT_X2 = ["AF_RSA"] + structural_3d_feature_cols + topology_feature_cols
FEAT_X2 = [c for c in FEAT_X2 if c in df_train.columns]

X1 = df_train[FEAT_X1].copy()
X2 = df_train[FEAT_X2].copy()
y_train_full = df_train["target"].astype(int).values
groups_full  = df_train["primary_accession"].astype(str).values

if len(esmc_df) > 0:
    esmc_cols_raw = sorted([c for c in esmc_df.columns if c.startswith("ESMC600_")])
    df_full = df_train.merge(esmc_df, on=["primary_accession", "residue_position"], how="inner")
    Xe_raw = df_full[esmc_cols_raw].values.astype(np.float32)
    scaler_esmc = StandardScaler()
    Xe_scaled = scaler_esmc.fit_transform(Xe_raw)
    N_PCA = 64
    pca_esmc = PCA(n_components=N_PCA, random_state=SEED)
    Xe_pca = pca_esmc.fit_transform(Xe_scaled)
    pca_col_names = [f"ESMC_PC{i:02d}" for i in range(N_PCA)]
    for i, c in enumerate(pca_col_names):
        df_full[c] = Xe_pca[:, i]
    FEAT_X3 = FEAT_X2 + pca_col_names
    X3 = df_full[FEAT_X3].copy()
    y_train_X3 = df_full["target"].astype(int).values
    groups_X3  = df_full["primary_accession"].astype(str).values
    joblib.dump(scaler_esmc, MODEL_DIR / "scaler_esmc.pkl")
    joblib.dump(pca_esmc,    MODEL_DIR / "pca_esmc.pkl")
    cumvar = np.cumsum(pca_esmc.explained_variance_ratio_)
    print(f"M3 (ESM-C PCA{N_PCA}): {Xe_raw.shape[1]} -> {N_PCA} dims, variance kept {cumvar[-1]*100:.1f}%")
else:
    X3 = None; y_train_X3 = None; groups_X3 = None; FEAT_X3 = None; pca_col_names = []
    df_full = pd.DataFrame()

print(f"X1 shape: {X1.shape}")
print(f"X2 shape: {X2.shape}")
if X3 is not None:
    print(f"X3 shape: {X3.shape}")

bundle = {
    "X1": X1, "X2": X2, "X3": X3,
    "y_X1X2": y_train_full, "y_X3": y_train_X3,
    "groups_X1X2": groups_full, "groups_X3": groups_X3,
    "FEAT_X1": FEAT_X1, "FEAT_X2": FEAT_X2, "FEAT_X3": FEAT_X3,
    "pca_col_names": pca_col_names,
    "df_train_meta": df_train[["primary_accession", "residue_position",
                               "residue_type", "source", "target"]].copy(),
}
atomic_pickle_dump(bundle, TABLE_DIR / "training_matrices.pkl")

## 12. CatBoost training

Identical CatBoost hyper-parameters across M1, M2, M3 so the comparison reflects only the feature set. 5-fold outer / 3-fold inner protein-aware CV. Inner CV chooses the operating threshold (balanced accuracy with MCC tiebreaker). Isotonic calibration is fit on pooled OOF scores.

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    balanced_accuracy_score, matthews_corrcoef, brier_score_loss,
)
from sklearn.isotonic import IsotonicRegression

OUTER, INNER = 5, 3
THRS = np.linspace(0.05, 0.95, 91)


def make_cb(seed):
    return CatBoostClassifier(
        iterations=800, depth=4, learning_rate=0.03,
        subsample=0.85, rsm=0.85, min_data_in_leaf=5, l2_leaf_reg=3.0,
        loss_function="Logloss", eval_metric="AUC",
        bootstrap_type="Bernoulli", random_seed=seed,
        thread_count=-1, verbose=False, allow_writing_files=False,
    )


def class_weights(y):
    n_pos, n_neg = int((y == 1).sum()), int((y == 0).sum())
    return np.where(y == 1, 1.0, n_pos / max(n_neg, 1)).astype(float)


def pick_threshold(y_true, p):
    best_t, best_s = 0.5, -np.inf
    for t in THRS:
        pred = (p >= t).astype(int)
        if len(np.unique(pred)) < 2:
            continue
        s = balanced_accuracy_score(y_true, pred) + 0.001 * matthews_corrcoef(y_true, pred)
        if s > best_s:
            best_s, best_t = s, float(t)
    return best_t


def nested_cv(X, y, groups, label):
    print(f"\n[{label}]  shape={X.shape}")
    outer = StratifiedGroupKFold(n_splits=OUTER, shuffle=True, random_state=SEED)
    oof_s = np.full(len(y), np.nan)
    oof_p = np.full(len(y), -1, dtype=int)
    folds = []
    for fi, (tr, te) in enumerate(outer.split(X, y, groups), 1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y[tr], y[te]; g_tr = groups[tr]
        inner_p = np.full(len(tr), np.nan)
        inner = StratifiedGroupKFold(n_splits=INNER, shuffle=True, random_state=SEED + fi)
        for itr, iva in inner.split(X_tr, y_tr, g_tr):
            mi = make_cb(SEED + fi * 100)
            mi.fit(X_tr.iloc[itr], y_tr[itr], sample_weight=class_weights(y_tr[itr]))
            inner_p[iva] = mi.predict_proba(X_tr.iloc[iva])[:, 1]
        valid_m = ~np.isnan(inner_p)
        thr = (pick_threshold(y_tr[valid_m], inner_p[valid_m])
               if len(np.unique(y_tr[valid_m])) >= 2 else 0.5)
        mo = make_cb(SEED + fi)
        mo.fit(X_tr, y_tr, sample_weight=class_weights(y_tr))
        score = mo.predict_proba(X_te)[:, 1]
        pred  = (score >= thr).astype(int)
        oof_s[te] = score; oof_p[te] = pred
        folds.append({
            "fold": fi, "thr": thr, "n_train": len(tr), "n_test": len(te),
            "roc_auc": roc_auc_score(y_te, score),
            "pr_auc":  average_precision_score(y_te, score),
            "bal_acc": balanced_accuracy_score(y_te, pred),
            "mcc":     matthews_corrcoef(y_te, pred),
            "brier":   brier_score_loss(y_te, score),
        })
    fold_df = pd.DataFrame(folds)
    pooled = {
        "roc_auc":  roc_auc_score(y, oof_s),
        "pr_auc":   average_precision_score(y, oof_s),
        "bal_acc":  balanced_accuracy_score(y, oof_p),
        "mcc":      matthews_corrcoef(y, oof_p),
        "brier":    brier_score_loss(y, oof_s),
        "mean_thr": float(fold_df["thr"].mean()),
    }
    print(fold_df.round(4).to_string(index=False))
    print({k: round(v, 4) for k, v in pooled.items()})
    return oof_s, oof_p, fold_df, pooled


oof_s1, oof_p1, fold1, pooled1 = nested_cv(X1, y_train_full, groups_full, "M1: rSASA only")
oof_s2, oof_p2, fold2, pooled2 = nested_cv(X2, y_train_full, groups_full, "M2: rSASA + features")
if X3 is not None:
    oof_s3, oof_p3, fold3, pooled3 = nested_cv(X3, y_train_X3, groups_X3, "M3: + ESM-C")
else:
    oof_s3 = oof_p3 = None
    pooled3 = {k: np.nan for k in ["roc_auc", "pr_auc", "bal_acc", "mcc", "brier", "mean_thr"]}

# Save OOF tables
for name, oof_s, oof_p, fdf, y_used, meta in [
    ("M1", oof_s1, oof_p1, fold1, y_train_full, df_train),
    ("M2", oof_s2, oof_p2, fold2, y_train_full, df_train),
]:
    pd.DataFrame({
        "primary_accession": meta["primary_accession"].values,
        "residue_position":  meta["residue_position"].values,
        "source":            meta["source"].values,
        "target":            y_used,
        "oof_score":         oof_s,
        "oof_pred":          oof_p,
    }).to_csv(MODEL_DIR / f"oof_{name}.csv", index=False)
    fdf.to_csv(MODEL_DIR / f"fold_results_{name}.csv", index=False)

if X3 is not None:
    pd.DataFrame({
        "primary_accession": df_full["primary_accession"].values,
        "residue_position":  df_full["residue_position"].values,
        "source":            df_full["source"].values,
        "target":            y_train_X3,
        "oof_score":         oof_s3,
        "oof_pred":          oof_p3,
    }).to_csv(MODEL_DIR / "oof_M3.csv", index=False)
    fold3.to_csv(MODEL_DIR / "fold_results_M3.csv", index=False)

# Refit final models and calibrators
w_full = class_weights(y_train_full)
cb1 = make_cb(SEED); cb1.fit(X1, y_train_full, sample_weight=w_full)
cb2 = make_cb(SEED); cb2.fit(X2, y_train_full, sample_weight=w_full)
joblib.dump(cb1, MODEL_DIR / "cb_final_M1.pkl")
joblib.dump(cb2, MODEL_DIR / "cb_final_M2.pkl")
joblib.dump(IsotonicRegression(out_of_bounds="clip").fit(oof_s1, y_train_full),
            MODEL_DIR / "iso_M1.pkl")
joblib.dump(IsotonicRegression(out_of_bounds="clip").fit(oof_s2, y_train_full),
            MODEL_DIR / "iso_M2.pkl")

if X3 is not None:
    cb3 = make_cb(SEED); cb3.fit(X3, y_train_X3, sample_weight=class_weights(y_train_X3))
    joblib.dump(cb3, MODEL_DIR / "cb_final_M3.pkl")
    joblib.dump(IsotonicRegression(out_of_bounds="clip").fit(oof_s3, y_train_X3),
                MODEL_DIR / "iso_M3.pkl")

## 13. Classifier benchmark

Same M2 and M3 feature sets, same protein-aware CV, run on Logistic Regression, Random Forest, KNN, MLP and SVM.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_curve

MODELS_TO_RUN = ["LR", "RF", "KNN", "MLP", "SVM"]


def make_estimator(name, seed=42):
    if name == "LR":
        return Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("scl", StandardScaler()),
                         ("clf", LogisticRegression(penalty="l2", C=1.0, solver="lbfgs",
                            max_iter=2000, class_weight="balanced", random_state=seed))])
    if name == "RF":
        return Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("clf", RandomForestClassifier(n_estimators=500, max_depth=None,
                            min_samples_leaf=5, class_weight="balanced",
                            n_jobs=-1, random_state=seed))])
    if name == "KNN":
        return Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("scl", StandardScaler()),
                         ("clf", KNeighborsClassifier(n_neighbors=15, weights="distance", n_jobs=-1))])
    if name == "MLP":
        return Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("scl", StandardScaler()),
                         ("clf", MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu",
                            solver="adam", alpha=1e-4, batch_size=128,
                            learning_rate_init=1e-3, max_iter=300,
                            early_stopping=True, validation_fraction=0.15,
                            n_iter_no_change=15, random_state=seed))])
    if name == "SVM":
        return Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("scl", StandardScaler()),
                         ("clf", SVC(kernel="rbf", C=1.0, gamma="scale",
                            class_weight="balanced", probability=True, random_state=seed))])
    raise ValueError(name)


def run_alt_model(name, X, y, groups, seed=SEED, n_splits=5):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_s = np.full(len(y), np.nan)
    folds = []
    for fi, (tr, te) in enumerate(cv.split(X, y, groups), 1):
        est = make_estimator(name, seed=seed + fi)
        try:
            est.fit(X.iloc[tr], y[tr])
            proba = est.predict_proba(X.iloc[te])[:, 1]
        except Exception:
            continue
        oof_s[te] = proba
        folds.append({"fold": fi, "n_train": len(tr), "n_test": len(te),
                      "roc_auc": roc_auc_score(y[te], proba),
                      "pr_auc":  average_precision_score(y[te], proba),
                      "brier":   brier_score_loss(y[te], proba)})
    fold_df = pd.DataFrame(folds)
    valid = ~np.isnan(oof_s)
    if valid.sum() < 10 or len(np.unique(y[valid])) < 2:
        return oof_s, None, fold_df, {"roc_auc": np.nan, "pr_auc": np.nan,
                                       "bal_acc": np.nan, "mcc": np.nan, "brier": np.nan}
    thr = pick_threshold(y[valid], oof_s[valid])
    oof_p = (oof_s >= thr).astype(int)
    pooled = {
        "roc_auc": roc_auc_score(y[valid], oof_s[valid]),
        "pr_auc":  average_precision_score(y[valid], oof_s[valid]),
        "bal_acc": balanced_accuracy_score(y[valid], oof_p[valid]),
        "mcc":     matthews_corrcoef(y[valid], oof_p[valid]),
        "brier":   brier_score_loss(y[valid], oof_s[valid]),
        "thr":     float(thr),
    }
    return oof_s, oof_p, fold_df, pooled


def benchmark_on(feature_set):
    if feature_set == "M2":
        X, y, groups = X2, y_train_full, groups_full
        cb_ref, cb_oof, cb_label, meta = pooled2, oof_s2, "CatBoost M2", df_train
    else:
        if X3 is None:
            print("X3 not available, skipping M3 benchmark"); return
        X, y, groups = X3, y_train_X3, groups_X3
        cb_ref, cb_oof, cb_label, meta = pooled3, oof_s3, "CatBoost M3", df_full

    print(f"\n=== Benchmark on {feature_set} ({X.shape}) ===")
    results = {}
    for name in MODELS_TO_RUN:
        oof_s, oof_p, fdf, pooled = run_alt_model(name, X, y, groups)
        results[name] = {"oof_score": oof_s, "pooled": pooled, "fold_df": fdf}
        print(f"  {name:4s} AUC={pooled.get('roc_auc', np.nan):.4f}  PR-AUC={pooled.get('pr_auc', np.nan):.4f}")
        pd.DataFrame({
            "primary_accession": meta["primary_accession"].values,
            "residue_position":  meta["residue_position"].values,
            "target":            y,
            "oof_score":         oof_s,
        }).to_csv(MODEL_DIR / f"oof_{name}_{feature_set}.csv", index=False)
        fdf.to_csv(MODEL_DIR / f"fold_results_{name}_{feature_set}.csv", index=False)

    rows = [{"model": name, **results[name]["pooled"]} for name in MODELS_TO_RUN]
    rows.append({"model": cb_label, **{k: cb_ref[k] for k in ["roc_auc", "pr_auc", "bal_acc", "mcc", "brier"]}})
    comp = pd.DataFrame(rows).sort_values("roc_auc", ascending=False).reset_index(drop=True)
    comp.to_csv(MODEL_DIR / f"alt_models_comparison_{feature_set}.csv", index=False)
    print()
    print(comp.round(4).to_string(index=False))

    # ROC overlay
    COLORS = {"LR": "#3498db", "RF": "#2ecc71", "KNN": "#e67e22",
              "MLP": "#9b59b6", "SVM": "#1abc9c", cb_label: "#e74c3c"}
    fig, ax = plt.subplots(figsize=(7, 5.5))
    for name in MODELS_TO_RUN:
        oof = results[name]["oof_score"]; vm = ~np.isnan(oof)
        if vm.sum() < 10: continue
        fpr, tpr, _ = roc_curve(y[vm], oof[vm])
        ax.plot(fpr, tpr, color=COLORS[name], lw=1.6,
                label=f"{name}  AUC={results[name]['pooled']['roc_auc']:.3f}", alpha=0.85)
    fpr, tpr, _ = roc_curve(y, cb_oof)
    ax.plot(fpr, tpr, color=COLORS[cb_label], lw=2.4,
            label=f"{cb_label}  AUC={cb_ref['roc_auc']:.3f}")
    ax.plot([0, 1], [0, 1], color="black", linestyle=":", alpha=0.4)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title(f"ROC curves on {feature_set} (pooled OOF)")
    ax.legend(loc="lower right", fontsize=9); ax.grid(alpha=0.25)
    fig.savefig(PLOT_DIR / f"benchmark_{feature_set}.png", dpi=150, bbox_inches="tight")
    fig.savefig(PLOT_DIR / f"benchmark_{feature_set}.svg", bbox_inches="tight")
    plt.show()


benchmark_on("M2")
benchmark_on("M3")

In [ ]:
# Numeri da stampare e confrontare con i valori del run vecchio
print(f"df_train shape: {df_train.shape}")
print(f"  pos: {int((df_train['target']==1).sum())}")
print(f"  neg: {int((df_train['target']==0).sum())}")
print(f"  proteins: {df_train['primary_accession'].nunique()}")
print(f"  source breakdown:")
print(df_train["source"].value_counts())

print(f"\nglygen_sites: {len(glygen_sites)} on {glygen_sites['primary_accession'].nunique()} proteins")

if "AF_RSA" in df_train.columns:
    print(f"\nrSASA stats on positives: mean={df_train.loc[df_train.target==1, 'AF_RSA'].mean():.3f}")
    print(f"rSASA stats on negatives: mean={df_train.loc[df_train.target==0, 'AF_RSA'].mean():.3f}")

## 14. SHAP feature importance

TreeExplainer on the refit CatBoost M3 model. Bar plot of mean absolute SHAP and beeswarm of value distributions, coloured by feature family.

In [ ]:
try:
    import shap
except ImportError:
    raise ImportError("Install shap first: pip install shap")

cb = joblib.load(MODEL_DIR / "cb_final_M3.pkl")
with open(TABLE_DIR / "training_matrices.pkl", "rb") as f:
    bundle = pickle.load(f)
X3        = bundle["X3"]
FEAT_X3   = bundle["FEAT_X3"]

explainer = shap.TreeExplainer(cb)
shap_values = explainer.shap_values(X3)
if shap_values.shape[1] == X3.shape[1] + 1:
    shap_values = shap_values[:, :-1]

mean_abs = np.abs(shap_values).mean(axis=0)
imp_df = (pd.DataFrame({"feature": FEAT_X3, "mean_abs_shap": mean_abs})
          .sort_values("mean_abs_shap", ascending=False).reset_index(drop=True))


def feat_group(name):
    if name == "AF_RSA":
        return "Structural (rSASA)"
    if "neighbor_count" in name or ("dist" in name and not name.startswith("Dist")):
        return "3D microenvironment"
    if "charge" in name:
        return "Electrostatics"
    if name.startswith("ESMC_PC"):
        return "ESM-C embedding"
    if name.startswith("Site_") or name.startswith("Dist_") or name.startswith("Rel_"):
        return "Topology (UniProt)"
    return "Other"


imp_df["group"] = imp_df["feature"].apply(feat_group)
imp_df.to_csv(TABLE_DIR / "shap_feature_importance.csv", index=False)
print(imp_df.head(20).to_string(index=False))

GROUP_COLORS = {
    "Structural (rSASA)":  "#e74c3c",
    "3D microenvironment": "#3498db",
    "Electrostatics":      "#9b59b6",
    "Topology (UniProt)":  "#27ae60",
    "ESM-C embedding":     "#f39c12",
    "Other":               "#7f8c8d",
}

TOP_N = 15
top_rows = imp_df.head(TOP_N).iloc[::-1].reset_index(drop=True)
top_feats = imp_df.head(TOP_N)["feature"].tolist()

fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[1.0, 1.4], wspace=0.35)

# Bar
ax = fig.add_subplot(gs[0, 0])
ax.barh(range(len(top_rows)), top_rows["mean_abs_shap"],
        color=[GROUP_COLORS[g] for g in top_rows["group"]], edgecolor="white", lw=0.6)
ax.set_yticks(range(len(top_rows)))
ax.set_yticklabels(top_rows["feature"], fontsize=9)
ax.set_xlabel("mean(|SHAP|)")
ax.set_title("Top features (mean absolute SHAP)")
ax.grid(axis="x", alpha=0.25)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in GROUP_COLORS.values()]
ax.legend(handles, GROUP_COLORS.keys(), loc="lower right", fontsize=8, framealpha=0.95)

# Beeswarm
ax = fig.add_subplot(gs[0, 1])
plt.sca(ax)
top_idx = [FEAT_X3.index(f) for f in top_feats]
shap.summary_plot(shap_values[:, top_idx], X3[top_feats],
                  feature_names=top_feats, plot_type="dot", show=False,
                  max_display=TOP_N, alpha=0.55)
ax.set_title("SHAP value distribution (beeswarm)")

fig.subplots_adjust(left=0.18, right=0.97, top=0.93, bottom=0.08)
fig.savefig(PLOT_DIR / "shap_M3.png", dpi=150, bbox_inches="tight")
fig.savefig(PLOT_DIR / "shap_M3.svg", bbox_inches="tight")
plt.show()

## 15. Cross-species validation: KPC47 (mouse)

Runs the full pipeline on the KPC47 mouse Spectronaut report (AlphaFold mouse PDBs, UniProt mouse, ESM-C, Top-N rSASA, feature recompute) and scores it with the M3 model. Outputs ROC and score distribution.

In [ ]:
KPC47_PATH = RAW_DATA_DIR / "KPC47_WTG1_WT_ERG1_4LB_SSL_ChRed_Report_Pivot_notNorm_Summ.csv"
KPC47_DIR  = OUT_DIR / "kpc47_validation"
KPC47_DIR.mkdir(parents=True, exist_ok=True)
KPC47_AF_DIR      = KPC47_DIR / "alphafold_pdb_mouse"
KPC47_AF_DIR.mkdir(parents=True, exist_ok=True)
KPC47_CACHES      = KPC47_DIR / "caches"; KPC47_CACHES.mkdir(parents=True, exist_ok=True)

if not KPC47_PATH.exists():
    raise FileNotFoundError(KPC47_PATH)
df_kpc = pd.read_csv(KPC47_PATH)
print(f"KPC47 report: {df_kpc.shape}")


def identify_columns_kpc47(df, drop_patterns=("WTG1",)):
    quant_cols = [c for c in df.columns if "MS2Quantity" in c]
    for pat in drop_patterns:
        quant_cols = [c for c in quant_cols if pat not in c]
    erg1_cols = [c for c in quant_cols if re.search(r"ERG1", c, flags=re.IGNORECASE)]
    ctrl_cols = [c for c in quant_cols if c not in erg1_cols
                 and re.search(r"WT|CTRL|GFP", c, flags=re.IGNORECASE)]
    return erg1_cols, ctrl_cols


e_cols, c_cols = identify_columns_kpc47(df_kpc)
df_kpc, n_dropped = drop_allnan_intensity_rows(df_kpc, e_cols + c_cols)
df_kpc_cls = classify_peptides(df_kpc, e_cols, c_cols)

mouse_accs = set()
for s in df_kpc_cls["PG.ProteinAccessions"].dropna().astype(str):
    for tok in re.split(r"[;,]", s):
        tok = tok.strip()
        if tok:
            mouse_accs.add(tok)
mouse_accs = sorted(mouse_accs)
print(f"Mouse accessions: {len(mouse_accs):,}")

In [ ]:
# Build AF cache for mouse proteins
AF_CACHE_MOUSE_PATH = KPC47_CACHES / "af_cache_mouse.pkl"
af_cache_mouse = load_pickle_if_valid(AF_CACHE_MOUSE_PATH)
if af_cache_mouse is None:
    af_cache_mouse = {}
    new, present, errors = download_alphafold(mouse_accs, KPC47_AF_DIR)
    print(f"AF mouse: downloaded={new}, present={present}, errors={len(errors)}")
    for acc in tqdm(mouse_accs, desc="SASA mouse"):
        fp = KPC47_AF_DIR / f"AF-{acc}-F1-model_v6.pdb"
        if not fp.exists():
            continue
        try:
            rec = parse_pdb_sasa_plddt(fp)
            if len(rec["sasa"]) > 0:
                af_cache_mouse[acc] = rec
        except Exception:
            continue
    atomic_pickle_dump(af_cache_mouse, AF_CACHE_MOUSE_PATH)


def get_rsasa_plddt_mouse(uid, pos, aa1):
    rec = af_cache_mouse.get(uid, {})
    abs_sasa = rec.get("sasa",  {}).get(pos)
    plddt    = rec.get("plddt", {}).get(pos)
    if abs_sasa is None or pd.isna(abs_sasa):
        return None, plddt
    aa3 = AA1_TO_AA3.get(aa1)
    if aa3 not in ("SER", "THR"):
        return None, plddt
    return min(float(abs_sasa) / TIEN_MAX[aa3], 1.0), plddt


# Build glycoform table with Top-N for KPC47 (reuse the function with af_cache_mouse swapped in)
_orig_get = globals()["get_rsasa_plddt"]
globals()["get_rsasa_plddt"] = get_rsasa_plddt_mouse
glyco_kpc = build_glycoform_topn(df_kpc_cls, "KPC47")
globals()["get_rsasa_plddt"] = _orig_get
print(f"KPC47 glycoforms: {len(glyco_kpc):,}")
glyco_kpc.to_csv(KPC47_DIR / "kpc47_glycoforms.csv", index=False)

In [ ]:
# Site-level KPC47 ground truth: ERG1 -> target=1, Shared -> target=0
glyco_kpc_valid = glyco_kpc[glyco_kpc["condition"].isin(
    ["ERG1_only", "shared_enriched", "shared_stable"])].copy()
glyco_kpc_valid["group"] = glyco_kpc_valid["condition"].map({
    "ERG1_only": "ERG1", "shared_enriched": "ERG1", "shared_stable": "Shared"})

sites_kpc = []
for _, r in glyco_kpc_valid.iterrows():
    key = r["topN_key"]
    if isinstance(key, str):
        key = eval(key)
    for pos in key:
        sites_kpc.append({"primary_accession": r["uniprot_id"],
                          "residue_position":  int(pos),
                          "group":             r["group"]})
sites_kpc_df = (pd.DataFrame(sites_kpc)
                .groupby(["primary_accession", "residue_position"])
                .apply(lambda g: pd.Series({
                    "target": int((g["group"] == "ERG1").any())
                }))
                .reset_index())


def aa1_from_mouse_cache(uid, pos):
    aa3 = af_cache_mouse.get(uid, {}).get("aa3", {}).get(int(pos))
    return AA3_TO_AA1.get(aa3) if aa3 else None


sites_kpc_df["residue_type"] = sites_kpc_df.apply(
    lambda r: aa1_from_mouse_cache(r["primary_accession"], r["residue_position"]), axis=1)
sites_kpc_df = sites_kpc_df[sites_kpc_df["residue_type"].isin(["S", "T"])].copy()
print(f"KPC47 sites: {len(sites_kpc_df):,}  pos={int(sites_kpc_df['target'].sum())}")

In [ ]:
# Compute the same features on KPC47 sites
af_rows_kpc = []
for _, r in sites_kpc_df.iterrows():
    uid, pos, aa = r["primary_accession"], int(r["residue_position"]), r["residue_type"]
    abs_sasa = af_cache_mouse.get(uid, {}).get("sasa", {}).get(pos)
    if abs_sasa is None or pd.isna(abs_sasa) or aa not in ("S", "T"):
        rsasa = np.nan
    else:
        rsasa = min(float(abs_sasa) / TIEN_MAX[AA1_TO_AA3[aa]], 1.0)
    af_rows_kpc.append({"primary_accession": uid, "residue_position": pos, "AF_RSA": rsasa})
af_kpc = pd.DataFrame(af_rows_kpc)

pdb_index_mouse = {}
for pdb_path in KPC47_AF_DIR.rglob("*.pdb"):
    m = re.search(r"AF-([A-Z0-9]+)-F1-model_v(\d+)\.pdb", pdb_path.name)
    if m:
        acc, ver = m.group(1), int(m.group(2))
        if acc not in pdb_index_mouse or ver > pdb_index_mouse[acc][1]:
            pdb_index_mouse[acc] = (pdb_path, ver)
pdb_index_mouse = {acc: p for acc, (p, _) in pdb_index_mouse.items()}

feat_3d_kpc = []
for acc, g in tqdm(sites_kpc_df.groupby("primary_accession"),
                   total=sites_kpc_df["primary_accession"].nunique(),
                   desc="3D KPC47"):
    if acc not in pdb_index_mouse:
        tmp = g[["primary_accession", "residue_position"]].copy()
        for c in structural_3d_feature_cols:
            tmp[c] = np.nan
        feat_3d_kpc.append(tmp); continue
    feat_3d_kpc.append(compute_3d_features(
        acc, g[["primary_accession", "residue_position"]], pdb_index_mouse[acc]))
feats_3d_kpc = pd.concat(feat_3d_kpc, ignore_index=True)

UNIPROT_KPC = KPC47_DIR / "uniprot_json"
UNIPROT_KPC.mkdir(parents=True, exist_ok=True)


def fetch_uniprot_mouse(acc):
    cache_path = UNIPROT_KPC / f"{acc}.json"
    if cache_path.exists():
        try:
            with open(cache_path) as f:
                return json.load(f)
        except Exception:
            pass
    try:
        r = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.json", timeout=30)
        if r.status_code != 200:
            return None
        data = r.json()
        with open(cache_path, "w") as f:
            json.dump(data, f)
        time.sleep(0.05)
        return data
    except Exception:
        return None


parsed_mouse = {}
for acc in tqdm(sorted(sites_kpc_df["primary_accession"].unique()),
                desc="UniProt mouse"):
    parsed_mouse[acc] = parse_uniprot_topology(fetch_uniprot_mouse(acc))

topo_kpc = []
for _, r in sites_kpc_df.iterrows():
    acc, pos = r["primary_accession"], int(r["residue_position"])
    p = parsed_mouse.get(acc, {"protein_length": np.nan, "signal_ranges": [], "tm_ranges": [],
                                "luminal_extra_ranges": [], "cytoplasmic_ranges": []})
    sig, tm, lum, cyt, plen = p["signal_ranges"], p["tm_ranges"], p["luminal_extra_ranges"], p["cytoplasmic_ranges"], p["protein_length"]
    signal_end = max([e for _, e in sig], default=np.nan)
    if pd.notna(signal_end):
        dist_signal = float(pos - signal_end); mature_start = int(signal_end) + 1
    else:
        dist_signal = np.nan; mature_start = 1
    rel_mature = (float((pos - mature_start) / max((plen - mature_start), 1))
                  if pd.notna(plen) and plen > 0 else np.nan)
    topo_kpc.append({
        "primary_accession": acc, "residue_position": pos,
        "Site_in_SignalPeptide": int(in_any_range(pos, sig)),
        "Site_in_TM_Domain":     int(in_any_range(pos, tm)),
        "Site_is_Luminal_Extra": int(in_any_range(pos, lum)),
        "Site_is_Cytoplasmic":   int(in_any_range(pos, cyt)),
        "Dist_from_Signal_End":  dist_signal,
        "Dist_to_Nearest_TM":    min_distance_to_ranges(pos, tm),
        "Rel_Pos_Mature_Protein": rel_mature,
        "Dist_to_TM_Boundary":   min_distance_to_range_boundary(pos, tm),
        "Rel_Pos_in_Luminal_Domain": rel_pos_in_ranges(pos, lum),
        "Dist_to_Luminal_Edge":  dist_to_luminal_edge(pos, lum),
    })
feats_topo_kpc = pd.DataFrame(topo_kpc)

df_kpc_features = (sites_kpc_df
                   .merge(af_kpc, on=["primary_accession", "residue_position"], how="left")
                   .merge(feats_3d_kpc, on=["primary_accession", "residue_position"], how="left")
                   .merge(feats_topo_kpc, on=["primary_accession", "residue_position"], how="left"))
df_kpc_features = df_kpc_features[df_kpc_features["AF_RSA"].notna()].reset_index(drop=True)
print(f"KPC47 with features: {df_kpc_features.shape}")

In [ ]:
# ESM-C for mouse proteins (reuses HF cache if model already downloaded)
ESMC_CACHE_MOUSE = KPC47_CACHES / "esmc_cache_mouse.pkl"
embedding_cache_m = load_pickle_if_valid(ESMC_CACHE_MOUSE)
if embedding_cache_m is None:
    embedding_cache_m = {}

needed_m = [f"{r.primary_accession}|{int(r.residue_position)}"
            for r in df_kpc_features.itertuples()]
needed_m = [k for k in needed_m if k not in embedding_cache_m]
print(f"Mouse ESM-C needed: {len(needed_m):,}")

if needed_m:
    import torch
    if torch.cuda.is_available():
        os.environ["HF_HOME"] = str(HF_DIR)
        from esm.models.esmc import ESMC
        from esm.sdk.api import ESMProtein, LogitsConfig
        sequence_map_m = {}
        for acc in {k.rsplit("|", 1)[0] for k in needed_m}:
            jp = UNIPROT_KPC / f"{acc}.json"
            if jp.exists():
                with open(jp) as f:
                    seq = json.load(f).get("sequence", {}).get("value")
                    if seq:
                        sequence_map_m[acc] = seq
        gc.collect(); torch.cuda.empty_cache()
        model_m = ESMC.from_pretrained("esmc_600m").to("cuda").half().eval()
        by_acc = {}
        for k in needed_m:
            acc, pos = k.rsplit("|", 1)
            by_acc.setdefault(acc, []).append(int(pos))
        for acc in tqdm(sorted(by_acc, key=lambda a: len(sequence_map_m.get(a, ""))),
                        desc="ESM-C mouse"):
            if acc not in sequence_map_m:
                continue
            seq = sequence_map_m[acc]; L = len(seq)
            if L > MAX_SEQ_LEN:
                continue
            try:
                with torch.no_grad():
                    pt = model_m.encode(ESMProtein(sequence=seq))
                    out = model_m.logits(pt, LogitsConfig(sequence=True, return_embeddings=True))
                emb = out.embeddings
                if isinstance(emb, torch.Tensor):
                    emb = emb.detach().cpu()
                if emb.ndim == 3:
                    emb = emb.squeeze(0)
                emb = emb.float().numpy(); nt = emb.shape[0]
                for pos in by_acc[acc]:
                    if 1 <= pos <= L:
                        idx = pos if nt >= L + 2 else (pos - 1 if nt == L else None)
                        if idx is None:
                            continue
                        embedding_cache_m[f"{acc}|{pos}"] = emb[idx].astype(np.float32)
                torch.cuda.empty_cache()
            except Exception:
                torch.cuda.empty_cache(); gc.collect()
        atomic_pickle_dump(embedding_cache_m, ESMC_CACHE_MOUSE)
        del model_m; gc.collect(); torch.cuda.empty_cache()
    else:
        print("CUDA not available, ESM-C skipped on mouse")

# Build matrix and score
rows_m = []
for _, r in df_kpc_features.iterrows():
    k = f"{r['primary_accession']}|{int(r['residue_position'])}"
    if k in embedding_cache_m:
        v = embedding_cache_m[k]
        rec = {"primary_accession": r["primary_accession"],
               "residue_position":  int(r["residue_position"])}
        for i, x in enumerate(v):
            rec[f"ESMC600_{i}"] = float(x)
        rows_m.append(rec)
esmc_kpc_df = pd.DataFrame(rows_m)
print(f"KPC47 with ESM-C: {esmc_kpc_df.shape}")

# Apply trained scaler + PCA
df_kpc_full = df_kpc_features.merge(esmc_kpc_df,
                                    on=["primary_accession", "residue_position"], how="inner")
esmc_cols_raw = sorted([c for c in esmc_kpc_df.columns if c.startswith("ESMC600_")])
scaler_esmc_loaded = joblib.load(MODEL_DIR / "scaler_esmc.pkl")
pca_esmc_loaded    = joblib.load(MODEL_DIR / "pca_esmc.pkl")
Xe = scaler_esmc_loaded.transform(df_kpc_full[esmc_cols_raw].values.astype(np.float32))
Xe_pca = pca_esmc_loaded.transform(Xe)
for i in range(Xe_pca.shape[1]):
    df_kpc_full[f"ESMC_PC{i:02d}"] = Xe_pca[:, i]

with open(TABLE_DIR / "training_matrices.pkl", "rb") as f:
    bundle = pickle.load(f)
FEAT_X3 = bundle["FEAT_X3"]
X_kpc = df_kpc_full[FEAT_X3]
y_kpc = df_kpc_full["target"].values.astype(int)

cb_final = joblib.load(MODEL_DIR / "cb_final_M3.pkl")
iso_M3   = joblib.load(MODEL_DIR / "iso_M3.pkl")
p_raw  = cb_final.predict_proba(X_kpc)[:, 1]
p_cal  = iso_M3.transform(p_raw)
df_kpc_full["p_raw"]  = p_raw
df_kpc_full["p_GALA"] = p_cal
df_kpc_full.to_csv(KPC47_DIR / "kpc47_scored.csv", index=False)

auc_kpc = roc_auc_score(y_kpc, p_cal)
pr_kpc  = average_precision_score(y_kpc, p_cal)
print(f"KPC47 cross-species: AUC={auc_kpc:.3f}  PR-AUC={pr_kpc:.3f}  n={len(y_kpc):,}")

# Plot
oof_train = pd.read_csv(MODEL_DIR / "oof_M3.csv")
fig = plt.figure(figsize=(13, 5.2))
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.28, width_ratios=[1, 1])

ax = fig.add_subplot(gs[0, 0])
fpr, tpr, _ = roc_curve(y_kpc, p_cal)
ax.plot(fpr, tpr, color="#e74c3c", lw=2.2, label=f"KPC47  AUC={auc_kpc:.3f}")
fpr_t, tpr_t, _ = roc_curve(oof_train["target"], oof_train["oof_score"])
auc_t = roc_auc_score(oof_train["target"], oof_train["oof_score"])
ax.plot(fpr_t, tpr_t, color="#7f8c8d", lw=1.6, linestyle="--",
        label=f"Training OOF  AUC={auc_t:.3f}")
ax.plot([0, 1], [0, 1], color="black", linestyle=":", alpha=0.4)
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("KPC47 cross-species ROC")
ax.legend(loc="lower right", fontsize=10); ax.grid(alpha=0.25)

ax = fig.add_subplot(gs[0, 1])
ax.hist(p_cal[y_kpc == 1], bins=30, color="#e74c3c", alpha=0.65,
        label=f"GALA (n={(y_kpc==1).sum():,})", density=True)
ax.hist(p_cal[y_kpc == 0], bins=30, color="#3498db", alpha=0.65,
        label=f"Golgi (n={(y_kpc==0).sum():,})", density=True)
ax.axvline(0.5, color="black", linestyle=":", alpha=0.5)
ax.set_xlabel("Calibrated p(GALA)"); ax.set_ylabel("Density")
ax.set_title("KPC47 score distribution")
ax.legend(loc="upper center", fontsize=10); ax.grid(alpha=0.2)

fig.savefig(PLOT_DIR / "kpc47_validation.png", dpi=150, bbox_inches="tight")
fig.savefig(PLOT_DIR / "kpc47_validation.svg", bbox_inches="tight")
plt.show()

## 16. Cross-tissue validation: KIC vs HMP (mouse)

KIC pancreatic tumor versus healthy mouse pancreas (HMP). Same Top-N pipeline; KIC tumour peptides act as the GALA-positive class, HMP healthy peptides as the Golgi reference.

In [ ]:
KIC_HMP_PATH = RAW_DATA_DIR / "2025_RB_KIC_HMP_10xLB_Report.xlsx"
KIC_HMP_DIR  = OUT_DIR / "kic_hmp_validation"
KIC_HMP_DIR.mkdir(parents=True, exist_ok=True)
KIC_HMP_AF_DIR = KIC_HMP_DIR / "alphafold_pdb_mouse"
KIC_HMP_AF_DIR.mkdir(parents=True, exist_ok=True)
KIC_HMP_CACHES = KIC_HMP_DIR / "caches"
KIC_HMP_CACHES.mkdir(parents=True, exist_ok=True)

if not KIC_HMP_PATH.exists():
    raise FileNotFoundError(KIC_HMP_PATH)
df_kh = pd.read_excel(KIC_HMP_PATH)
print(f"KIC vs HMP report: {df_kh.shape}")


def identify_kic_hmp(df):
    quant_cols = [c for c in df.columns if "MS2Quantity" in c]
    kic_cols = [c for c in quant_cols if re.search(r"_KIC\d", c)]
    hmp_cols = [c for c in quant_cols if re.search(r"_HMP\d", c)]
    return kic_cols, hmp_cols


def classify_kic_hmp(df, kic_cols, hmp_cols, log2fc_thr=LOGFC_THRESHOLD, min_frac=MIN_FRAC):
    out = df.copy()
    min_kic = math.ceil(len(kic_cols) * min_frac)
    min_hmp = math.ceil(len(hmp_cols) * min_frac)
    out["present_KIC"] = out[kic_cols].notna().sum(axis=1) >= min_kic
    out["present_HMP"] = out[hmp_cols].notna().sum(axis=1) >= min_hmp
    out["mean_KIC"]    = out[kic_cols].mean(axis=1)
    out["mean_HMP"]    = out[hmp_cols].mean(axis=1)
    pseudo = 1.0
    out["log2FC"] = np.log2((out["mean_KIC"].fillna(0) + pseudo) /
                            (out["mean_HMP"].fillna(0) + pseudo))
    cond = [
        out["present_KIC"] & ~out["present_HMP"],
        out["present_KIC"] &  out["present_HMP"] & (out["log2FC"] >  log2fc_thr),
        out["present_KIC"] &  out["present_HMP"] & (out["log2FC"] <= log2fc_thr),
    ]
    out["condition"] = np.select(cond,
        ["ERG1_only", "shared_enriched", "shared_stable"], default="low_confidence")
    return out


k_cols, h_cols = identify_kic_hmp(df_kh)
df_kh, _ = drop_allnan_intensity_rows(df_kh, k_cols + h_cols)
df_kh_cls = classify_kic_hmp(df_kh, k_cols, h_cols)

# Mouse accessions
kh_accs = set()
for s in df_kh_cls["PG.ProteinAccessions"].dropna().astype(str):
    for tok in re.split(r"[;,]", s):
        tok = tok.strip()
        if tok:
            kh_accs.add(tok)
kh_accs = sorted(kh_accs)
print(f"KIC vs HMP mouse accessions: {len(kh_accs):,}")

In [ ]:
# AF cache for KIC_HMP (overlap with KPC47 mouse cache when possible)
AF_CACHE_KH_PATH = KIC_HMP_CACHES / "af_cache_kic_hmp.pkl"
af_cache_kh = load_pickle_if_valid(AF_CACHE_KH_PATH)
if af_cache_kh is None:
    af_cache_kh = {}
    new, present, errors = download_alphafold(kh_accs, KIC_HMP_AF_DIR)
    print(f"AF KIC_HMP: downloaded={new}, present={present}, errors={len(errors)}")
    for acc in tqdm(kh_accs, desc="SASA KIC_HMP"):
        fp = KIC_HMP_AF_DIR / f"AF-{acc}-F1-model_v6.pdb"
        if not fp.exists():
            continue
        try:
            rec = parse_pdb_sasa_plddt(fp)
            if len(rec["sasa"]) > 0:
                af_cache_kh[acc] = rec
        except Exception:
            continue
    atomic_pickle_dump(af_cache_kh, AF_CACHE_KH_PATH)


def get_rsasa_plddt_kh(uid, pos, aa1):
    rec = af_cache_kh.get(uid, {})
    abs_sasa = rec.get("sasa",  {}).get(pos)
    plddt    = rec.get("plddt", {}).get(pos)
    if abs_sasa is None or pd.isna(abs_sasa):
        return None, plddt
    aa3 = AA1_TO_AA3.get(aa1)
    if aa3 not in ("SER", "THR"):
        return None, plddt
    return min(float(abs_sasa) / TIEN_MAX[aa3], 1.0), plddt


_orig_get = globals()["get_rsasa_plddt"]
globals()["get_rsasa_plddt"] = get_rsasa_plddt_kh
glyco_kh = build_glycoform_topn(df_kh_cls, "KIC_HMP")
globals()["get_rsasa_plddt"] = _orig_get
print(f"KIC vs HMP glycoforms: {len(glyco_kh):,}")
glyco_kh.to_csv(KIC_HMP_DIR / "kic_hmp_glycoforms.csv", index=False)

# Site-level ground truth
glyco_kh_valid = glyco_kh[glyco_kh["condition"].isin(
    ["ERG1_only", "shared_enriched", "shared_stable"])].copy()
glyco_kh_valid["group"] = glyco_kh_valid["condition"].map({
    "ERG1_only": "ERG1", "shared_enriched": "ERG1", "shared_stable": "Shared"})

sites_kh = []
for _, r in glyco_kh_valid.iterrows():
    key = r["topN_key"]
    if isinstance(key, str):
        key = eval(key)
    for pos in key:
        sites_kh.append({"primary_accession": r["uniprot_id"],
                         "residue_position":  int(pos),
                         "group":             r["group"]})
sites_kh_df = (pd.DataFrame(sites_kh)
               .groupby(["primary_accession", "residue_position"])
               .apply(lambda g: pd.Series({"target": int((g["group"] == "ERG1").any())}))
               .reset_index())
sites_kh_df["residue_type"] = sites_kh_df.apply(
    lambda r: AA3_TO_AA1.get(af_cache_kh.get(r["primary_accession"], {}).get("aa3", {}).get(int(r["residue_position"]))),
    axis=1)
sites_kh_df = sites_kh_df[sites_kh_df["residue_type"].isin(["S", "T"])].copy()
print(f"KIC vs HMP sites: {len(sites_kh_df):,}  pos={int(sites_kh_df['target'].sum())}")

In [ ]:
# Features
af_rows_kh = []
for _, r in sites_kh_df.iterrows():
    uid, pos, aa = r["primary_accession"], int(r["residue_position"]), r["residue_type"]
    abs_sasa = af_cache_kh.get(uid, {}).get("sasa", {}).get(pos)
    if abs_sasa is None or pd.isna(abs_sasa) or aa not in ("S", "T"):
        rsasa = np.nan
    else:
        rsasa = min(float(abs_sasa) / TIEN_MAX[AA1_TO_AA3[aa]], 1.0)
    af_rows_kh.append({"primary_accession": uid, "residue_position": pos, "AF_RSA": rsasa})
af_kh = pd.DataFrame(af_rows_kh)

pdb_index_kh = {}
for pdb_path in KIC_HMP_AF_DIR.rglob("*.pdb"):
    m = re.search(r"AF-([A-Z0-9]+)-F1-model_v(\d+)\.pdb", pdb_path.name)
    if m:
        acc, ver = m.group(1), int(m.group(2))
        if acc not in pdb_index_kh or ver > pdb_index_kh[acc][1]:
            pdb_index_kh[acc] = (pdb_path, ver)
pdb_index_kh = {acc: p for acc, (p, _) in pdb_index_kh.items()}

feat_3d_kh = []
for acc, g in tqdm(sites_kh_df.groupby("primary_accession"),
                   total=sites_kh_df["primary_accession"].nunique(),
                   desc="3D KIC_HMP"):
    if acc not in pdb_index_kh:
        tmp = g[["primary_accession", "residue_position"]].copy()
        for c in structural_3d_feature_cols:
            tmp[c] = np.nan
        feat_3d_kh.append(tmp); continue
    feat_3d_kh.append(compute_3d_features(
        acc, g[["primary_accession", "residue_position"]], pdb_index_kh[acc]))
feats_3d_kh = pd.concat(feat_3d_kh, ignore_index=True)

UNIPROT_KH = KIC_HMP_DIR / "uniprot_json"
UNIPROT_KH.mkdir(parents=True, exist_ok=True)


def fetch_uniprot_kh(acc):
    cache_path = UNIPROT_KH / f"{acc}.json"
    if cache_path.exists():
        try:
            with open(cache_path) as f:
                return json.load(f)
        except Exception:
            pass
    try:
        r = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.json", timeout=30)
        if r.status_code != 200:
            return None
        data = r.json()
        with open(cache_path, "w") as f:
            json.dump(data, f)
        time.sleep(0.05)
        return data
    except Exception:
        return None


parsed_kh = {}
for acc in tqdm(sorted(sites_kh_df["primary_accession"].unique()), desc="UniProt KIC_HMP"):
    parsed_kh[acc] = parse_uniprot_topology(fetch_uniprot_kh(acc))

topo_kh_rows = []
for _, r in sites_kh_df.iterrows():
    acc, pos = r["primary_accession"], int(r["residue_position"])
    p = parsed_kh.get(acc, {"protein_length": np.nan, "signal_ranges": [], "tm_ranges": [],
                             "luminal_extra_ranges": [], "cytoplasmic_ranges": []})
    sig, tm, lum, cyt, plen = p["signal_ranges"], p["tm_ranges"], p["luminal_extra_ranges"], p["cytoplasmic_ranges"], p["protein_length"]
    signal_end = max([e for _, e in sig], default=np.nan)
    if pd.notna(signal_end):
        dist_signal = float(pos - signal_end); mature_start = int(signal_end) + 1
    else:
        dist_signal = np.nan; mature_start = 1
    rel_mature = (float((pos - mature_start) / max((plen - mature_start), 1))
                  if pd.notna(plen) and plen > 0 else np.nan)
    topo_kh_rows.append({
        "primary_accession": acc, "residue_position": pos,
        "Site_in_SignalPeptide": int(in_any_range(pos, sig)),
        "Site_in_TM_Domain":     int(in_any_range(pos, tm)),
        "Site_is_Luminal_Extra": int(in_any_range(pos, lum)),
        "Site_is_Cytoplasmic":   int(in_any_range(pos, cyt)),
        "Dist_from_Signal_End":  dist_signal,
        "Dist_to_Nearest_TM":    min_distance_to_ranges(pos, tm),
        "Rel_Pos_Mature_Protein": rel_mature,
        "Dist_to_TM_Boundary":   min_distance_to_range_boundary(pos, tm),
        "Rel_Pos_in_Luminal_Domain": rel_pos_in_ranges(pos, lum),
        "Dist_to_Luminal_Edge":  dist_to_luminal_edge(pos, lum),
    })
feats_topo_kh = pd.DataFrame(topo_kh_rows)

df_kh_features = (sites_kh_df
                  .merge(af_kh,        on=["primary_accession", "residue_position"], how="left")
                  .merge(feats_3d_kh,  on=["primary_accession", "residue_position"], how="left")
                  .merge(feats_topo_kh, on=["primary_accession", "residue_position"], how="left"))
df_kh_features = df_kh_features[df_kh_features["AF_RSA"].notna()].reset_index(drop=True)

# ESM-C for KIC_HMP
ESMC_CACHE_KH = KIC_HMP_CACHES / "esmc_cache_kic_hmp.pkl"
embedding_cache_kh = load_pickle_if_valid(ESMC_CACHE_KH)
if embedding_cache_kh is None:
    embedding_cache_kh = {}

needed_kh = [f"{r.primary_accession}|{int(r.residue_position)}"
             for r in df_kh_features.itertuples()]
needed_kh = [k for k in needed_kh if k not in embedding_cache_kh]

if needed_kh:
    import torch
    if torch.cuda.is_available():
        os.environ["HF_HOME"] = str(HF_DIR)
        from esm.models.esmc import ESMC
        from esm.sdk.api import ESMProtein, LogitsConfig
        seq_map = {}
        for acc in {k.rsplit("|", 1)[0] for k in needed_kh}:
            jp = UNIPROT_KH / f"{acc}.json"
            if jp.exists():
                with open(jp) as f:
                    seq = json.load(f).get("sequence", {}).get("value")
                    if seq:
                        seq_map[acc] = seq
        gc.collect(); torch.cuda.empty_cache()
        m_esm = ESMC.from_pretrained("esmc_600m").to("cuda").half().eval()
        by_acc = {}
        for k in needed_kh:
            acc, pos = k.rsplit("|", 1)
            by_acc.setdefault(acc, []).append(int(pos))
        for acc in tqdm(sorted(by_acc, key=lambda a: len(seq_map.get(a, ""))),
                        desc="ESM-C KIC_HMP"):
            if acc not in seq_map:
                continue
            seq = seq_map[acc]; L = len(seq)
            if L > MAX_SEQ_LEN:
                continue
            try:
                with torch.no_grad():
                    pt = m_esm.encode(ESMProtein(sequence=seq))
                    out = m_esm.logits(pt, LogitsConfig(sequence=True, return_embeddings=True))
                emb = out.embeddings
                if isinstance(emb, torch.Tensor):
                    emb = emb.detach().cpu()
                if emb.ndim == 3:
                    emb = emb.squeeze(0)
                emb = emb.float().numpy(); nt = emb.shape[0]
                for pos in by_acc[acc]:
                    if 1 <= pos <= L:
                        idx = pos if nt >= L + 2 else (pos - 1 if nt == L else None)
                        if idx is None:
                            continue
                        embedding_cache_kh[f"{acc}|{pos}"] = emb[idx].astype(np.float32)
                torch.cuda.empty_cache()
            except Exception:
                torch.cuda.empty_cache(); gc.collect()
        atomic_pickle_dump(embedding_cache_kh, ESMC_CACHE_KH)
        del m_esm; gc.collect(); torch.cuda.empty_cache()

rows_kh = []
for _, r in df_kh_features.iterrows():
    k = f"{r['primary_accession']}|{int(r['residue_position'])}"
    if k in embedding_cache_kh:
        v = embedding_cache_kh[k]
        rec = {"primary_accession": r["primary_accession"],
               "residue_position":  int(r["residue_position"])}
        for i, x in enumerate(v):
            rec[f"ESMC600_{i}"] = float(x)
        rows_kh.append(rec)
esmc_kh_df = pd.DataFrame(rows_kh)

df_kh_full = df_kh_features.merge(esmc_kh_df,
                                  on=["primary_accession", "residue_position"], how="inner")
esmc_cols_raw = sorted([c for c in esmc_kh_df.columns if c.startswith("ESMC600_")])
Xe_kh = scaler_esmc_loaded.transform(df_kh_full[esmc_cols_raw].values.astype(np.float32))
Xe_kh_pca = pca_esmc_loaded.transform(Xe_kh)
for i in range(Xe_kh_pca.shape[1]):
    df_kh_full[f"ESMC_PC{i:02d}"] = Xe_kh_pca[:, i]

X_kh = df_kh_full[FEAT_X3]
y_kh = df_kh_full["target"].values.astype(int)
p_kh = iso_M3.transform(cb_final.predict_proba(X_kh)[:, 1])
df_kh_full["p_GALA"] = p_kh
df_kh_full.to_csv(KIC_HMP_DIR / "kic_hmp_scored.csv", index=False)

auc_kh = roc_auc_score(y_kh, p_kh)
pr_kh  = average_precision_score(y_kh, p_kh)
print(f"KIC vs HMP: AUC={auc_kh:.3f}  PR-AUC={pr_kh:.3f}  n={len(y_kh):,}")

# Plot
fig = plt.figure(figsize=(13, 5.2))
gs = gridspec.GridSpec(1, 2, figure=fig, wspace=0.28)

ax = fig.add_subplot(gs[0, 0])
fpr, tpr, _ = roc_curve(y_kh, p_kh)
ax.plot(fpr, tpr, color="#e74c3c", lw=2.2, label=f"KIC vs HMP  AUC={auc_kh:.3f}")
fpr_t, tpr_t, _ = roc_curve(oof_train["target"], oof_train["oof_score"])
auc_t = roc_auc_score(oof_train["target"], oof_train["oof_score"])
ax.plot(fpr_t, tpr_t, color="#7f8c8d", lw=1.6, linestyle="--",
        label=f"Training OOF  AUC={auc_t:.3f}")
ax.plot([0, 1], [0, 1], color="black", linestyle=":", alpha=0.4)
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("KIC vs HMP ROC")
ax.legend(loc="lower right", fontsize=10); ax.grid(alpha=0.25)

ax = fig.add_subplot(gs[0, 1])
ax.hist(p_kh[y_kh == 1], bins=30, color="#e74c3c", alpha=0.65,
        label=f"KIC (n={(y_kh==1).sum():,})", density=True)
ax.hist(p_kh[y_kh == 0], bins=30, color="#3498db", alpha=0.65,
        label=f"HMP (n={(y_kh==0).sum():,})", density=True)
ax.axvline(0.5, color="black", linestyle=":", alpha=0.5)
ax.set_xlabel("Calibrated p(GALA)"); ax.set_ylabel("Density")
ax.set_title("KIC vs HMP score distribution")
ax.legend(loc="upper center", fontsize=10); ax.grid(alpha=0.2)

fig.savefig(PLOT_DIR / "kic_hmp_validation.png", dpi=150, bbox_inches="tight")
fig.savefig(PLOT_DIR / "kic_hmp_validation.svg", bbox_inches="tight")
plt.show()